# Predicción de defectos para QA: validación entre proyectos y priorización por esfuerzo

**Tesis T2 — Grupo B, UNMSM**  
**Objetivo:** entrenar y evaluar el clasificador SVM E6, contrastarlo con otras recetas y, especialmente, comprobar si la **priorización consciente del esfuerzo** mejora el trabajo de inspección de software frente a la regla simple **ManualUp**.

### Qué incluye este cuaderno

1. Descarga verificable de cinco proyectos (CM1, JM1, KC1, KC2 y PC1) con hashes SHA-256.
2. Auditoría de duplicados y etiquetas contradictorias; cohorte principal y análisis de sensibilidad.
3. Evaluación Leave-One-Project-Out de los modelos E6 y E7; métricas F1, AP y AUC.
4. Curvas y comparaciones de priorización bajo presupuestos de LOC y de módulos.
5. **Estrategia recomendada por el asesor:** transformar el margen SVM de E6 en una probabilidad calibrada sin usar las etiquetas del proyecto de prueba, y ordenar por **probabilidad / LOC**.
6. Intervalos bootstrap por proyecto, diferencias con ManualUp, gráficos y un archivo ZIP con los resultados y el código.

### Alcance e integridad

- **E6** (ADASYN + selección por información mutua) es el clasificador ya utilizado por el prototipo Streamlit; ejecutar aquí no reemplaza su modelo.
- Los CSV son un espejo público de conjuntos históricos de defectos. Se verifican los hashes, pero la **procedencia académica y edición exacta de los datasets** todavía deben documentarse.
- Se utilizan las **21 métricas precomputadas** del estudio. Este cuaderno no extrae automáticamente métricas de un repositorio fuente.
- Los mismos cinco proyectos ya han sido examinados durante el desarrollo; por tanto, la nueva estrategia por esfuerzo es una **evaluación adicional exploratoria**, no una confirmación independiente ni un análisis preregistrado.
- Se conservan y reportan los resultados negativos. **No se ajustan metas o criterios después de ver la nueva salida** para poder afirmar una mejora.

**Cómo ejecutar:** Abre en Google Colab, elige CPU y selecciona *Entorno de ejecución → Ejecutar todas*. Se generan los scripts Python en las celdas siguientes; no se necesitan notebooks anteriores ni archivos de experimentos previos. La descarga de los CSV requiere conexión; si falla, se solicitará un ZIP con los cinco CSV originales y se verificarán sus hashes. Si la instalación de dependencias pide reiniciar, reinicia y ejecuta de nuevo.

## 0. Instalar dependencias del experimento

Estas dependencias se fijan para que la ejecución sea reproducible. Colab requiere acceso a internet para instalar las bibliotecas y descargar los archivos CSV. Si pide reiniciar el entorno después de la instalación, reinícialo y ejecuta todas las celdas otra vez. No reemplaces automáticamente el archivo `.joblib` que usa Streamlit.


In [ ]:
%pip -q install "scikit-learn==1.6.1" "imbalanced-learn==0.14.2" "xgboost==3.4.1" "pandas>=2.1" "numpy>=1.26" "scipy>=1.11" "joblib>=1.3" "matplotlib>=3.7"

## 1. Configuración de la ejecución

**Ejecución predeterminada para responder la observación del profesor:** auditar datos y ejecutar el nuevo ranking con **E6 calibrado / LOC frente a ManualUp**. Las comparaciones completas E6/E7 y las recetas históricas siguen incluidas en el código y pueden activarse, pero se dejan apagadas por defecto para no repetir entrenamientos costosos que ya cuentan con resultados anteriores.

La nueva estrategia reentrena modelos en cada partición fuente para calibrar sin observar el proyecto de prueba. No se cambian los umbrales después de obtener resultados. El ZIP final contiene los scripts y las cifras generadas durante esta ejecución; las tablas referenciales se identifican como antecedentes.

In [ ]:
from pathlib import Path
import json, os, hashlib, zipfile, urllib.request, shutil, sys
import numpy as np
import pandas as pd
from datetime import datetime, timezone

DIRECTORIO = Path('/content/T2_CPDP_QA')
DIRECTORIO.mkdir(parents=True,exist_ok=True)
DATA_DIR = DIRECTORIO / '01_datos_originales'
OUT_DIR = DIRECTORIO / '02_resultados'
# Evita incluir resultados obsoletos al reejecutar en la misma sesión.
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True,exist_ok=True)

RUN_FIJO_E6_E7 = False          # 5 proyectos × 2 modelos × 2 cohortes (principal y sensibilidad)
REPRODUCIR_SELECCIONES_PREVIAS = False  # 5 proyectos × 2 cohortes; usa selecciones previas incluidas aquí
EXPORTAR_DOS_CANDIDATOS = False
DESCARGAR_ZIP_AL_FINAL = True
REPETIR_BUSQUEDA_COMPLETA_27 = False   # Opcional: vuelve a buscar 27 configuraciones desde cero
SEED = 42
print('Carpeta de trabajo:',DIRECTORIO)


## 2. Código central del entrenamiento (incluido íntegramente)

Esta celda crea `cpdp_core.py` **directamente a partir del contenido de este notebook**. Define las 21 métricas, el procesamiento de datos, los escenarios de muestreo/selección, los algoritmos SVM/RF/XGBoost y el procedimiento Leave-One-Project-Out (LOPO). No necesita importar archivos Python de otro ZIP o repositorio.

**Control científico:** la separación por proyecto y la preparación dentro de cada entrenamiento ayudan a evitar fugas. El conjunto de cinco proyectos no equivale a una prueba con datos inéditos de una sexta organización.


In [ ]:
%%writefile cpdp_core.py
"""PDS tesis — validación ENTRE proyectos NASA/PROMISE, nueva ejecución completamente independiente.

Experimento educativo auditable: no es garantía de defectos de un proyecto nuevo.
Únicamente métricas históricas compatibles, no extrae código fuente.
"""
from __future__ import annotations
import hashlib, json, os, platform, sys, time
from datetime import datetime, timezone
from functools import partial
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif, SelectFromModel
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (f1_score, recall_score, precision_score, accuracy_score,
    average_precision_score, roc_auc_score, confusion_matrix)
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, ADASYN
from xgboost import XGBClassifier
import joblib

SEED = 42
EXPERIMENT_VERSION = 'CPDP_T2_ESTUDIO_QA'
PROJECTS = ('CM1','JM1','KC1','KC2','PC1')
LABELS = {'CM1':'defects','JM1':'defects','KC1':'defects','KC2':'problems','PC1':'defects'}
SOURCE_BASE = 'https://raw.githubusercontent.com/ApoorvaKrisna/NASA-promise-dataset-repository/main'
# Orden de las 21 métricas; preservado en la exportación e inferencia. Se valida antes de unir.
FEATURES = (
    'loc','v(g)','ev(g)','iv(g)','n','v','l','d','i','e','b','t',
    'lOCode','lOComment','lOBlank','locCodeAndComment',
    'uniq_Op','uniq_Opnd','total_Op','total_Opnd','branchCount'
)
# Alias explícito a documentar. OJO: no implica equivalencia verificada por el autor.
ALIASES = {'locodeandcomment':'loccodeandcomment'}
SCENARIOS = (
    ('E0_BASELINE','NONE','NONE'),
    ('E1_SMOTE','SMOTE','NONE'),
    ('E2_ADASYN','ADASYN','NONE'),
    ('E3_MI','NONE','MI'),
    ('E4_RF_FS','NONE','RF'),
    ('E5_SMOTE_MI','SMOTE','MI'),
    ('E6_ADASYN_MI','ADASYN','MI'),
    ('E7_SMOTE_RF','SMOTE','RF'),
    ('E8_ADASYN_RF','ADASYN','RF'),
)
ALGORITHMS = ('SVM','Random Forest','XGBoost')
CANDIDATES = tuple((s[0],a) for s in SCENARIOS for a in ALGORITHMS)


def sha256(path):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for part in iter(lambda:f.read(1024*1024),b''):
            h.update(part)
    return h.hexdigest()


def jdump(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,allow_nan=False,
        default=lambda x:x.item() if isinstance(x,np.generic) else str(x)),encoding='utf-8')
    os.replace(tmp,path)


def standard_key(s):
    return str(s).strip().casefold()


def harmonize_features(raw,project,*, allow_extra=False):
    """Mapeo verificable por encabezados, nunca descartar columnas silenciosamente."""
    raw=raw.copy()
    found={}; alias_logs=[]
    for col in raw.columns:
        k=standard_key(col)
        if k in ALIASES:
            alias_logs.append({'proyecto':project,'columna_original':str(col),
                'equivalencia_propuesta':'locCodeAndComment','pendiente_verificacion_documental':True})
            k=ALIASES[k]
        if k in found:
            raise ValueError(f'{project}: colisión tras normalizar columnas {found[k]} y {col}')
        found[k]=col
    desired={standard_key(x):x for x in FEATURES}
    missing=sorted(set(desired)-set(found))
    extras=sorted(set(found)-set(desired))
    if missing or (extras and not allow_extra):
        raise ValueError(f'{project}: esquema incompatible: faltantes={missing}, extras={extras}')
    out=pd.DataFrame(index=raw.index)
    for norm,canonical in desired.items():
        vals=raw[found[norm]].replace(['?','NA','NaN',''],np.nan)
        num=pd.to_numeric(vals,errors='coerce')
        invalid=vals.notna() & num.isna()
        if invalid.any():
            raise ValueError(f'{project}: caracteres no numéricos en {found[norm]}')
        if np.isinf(num.to_numpy(dtype=float,na_value=np.nan)).any():
            raise ValueError(f'{project}: valores infinitos en {found[norm]}')
        out[canonical]=num.astype(float)
    if out.empty: raise ValueError(f'{project}: dataset vacío')
    return out,alias_logs


def load_project(path,name):
    """Solo admite etiqueta histórica conocida por proyecto y 21 métricas esperadas."""
    if name not in PROJECTS: raise ValueError(name)
    raw=pd.read_csv(path,low_memory=False)
    if raw.empty or raw.columns.duplicated().any(): raise ValueError(f'{name}: CSV vacío o cabeceras repetidas')
    cols=[c for c in raw if standard_key(c)==LABELS[name]]
    if len(cols)!=1: raise ValueError(f'{name}: etiqueta esperada={LABELS[name]}, detectadas={list(raw.columns)}')
    target=cols[0]
    labels=raw[target].astype('string').str.strip().str.casefold()
    mapping={'false':0,'true':1,'0':0,'1':1,'0.0':0,'1.0':1,'no':0,'yes':1}
    unexpected=sorted(set(labels.dropna())-set(mapping))
    if unexpected or labels.isna().any():
        raise ValueError(f'{name}: valores de etiqueta desconocidos={unexpected}')
    y=labels.map(mapping).astype(int).reset_index(drop=True)
    if y.nunique()!=2: raise ValueError(f'{name}: se necesitan ambas clases')
    X,alias_logs=harmonize_features(raw.drop(columns=[target]),name)
    X=X.reset_index(drop=True)
    fingerprint=pd.util.hash_pandas_object(X,index=False)
    audit={'proyecto':name,'filas':len(y),'positivos':int(y.sum()),'negativos':int((y==0).sum()),
        'prevalencia':float(y.mean()),'variables':len(FEATURES),
        'etiqueta_original':target,'etiqueta_binaria':'1=defecto_historico_registrado',
        'sha256':sha256(path),'duplicados_fila_completa':int(raw.duplicated().sum()),
        'duplicados_vector_caracteristicas':int(fingerprint.duplicated().sum()),
        'faltantes':int(X.isna().sum().sum()),'alias_pendientes':alias_logs,
        'columnas_originales':list(map(str,raw.columns)),
        'procedencia_declarada':f'{SOURCE_BASE}/{name.lower()}.csv',
        'verificacion_documental_realizada':False}
    return X,y,audit


def load_all(raw_dir,names=PROJECTS):
    data={};audits=[]
    for name in names:
        p=Path(raw_dir)/f'{name}.csv'
        if not p.exists(): raise FileNotFoundError(f'{name}: falta {p}')
        x,y,a=load_project(p,name)
        data[name]=(x,y)
        audits.append(a)
    return data,audits


def overlap_audit(data):
    hashes={};pairs=[]
    for p,(x,y) in data.items():
        hashes[p]=pd.util.hash_pandas_object(x,index=False).to_numpy(dtype=np.uint64)
    for i,p in enumerate(data):
        for q in list(data)[i+1:]:
            hs1=set(hashes[p].tolist());hs2=set(hashes[q].tolist());common=hs1&hs2
            pairs.append({'proyecto_a':p,'proyecto_b':q,'vectores_identicos':len(common),
               'filas_de_a_coincidentes':int(np.isin(hashes[p],list(common)).sum()),
               'filas_de_b_coincidentes':int(np.isin(hashes[q],list(common)).sum())})
    return pd.DataFrame(pairs)


def pack(data,projects):
    """Cada ejemplo pertenece a su proyecto; sin derivar información de ningún test."""
    return (pd.concat([data[p][0] for p in projects],ignore_index=True),
            pd.concat([data[p][1] for p in projects],ignore_index=True))


def make_pipeline(scenario_id,algorithm):
    _,balance,select=next(v for v in SCENARIOS if v[0]==scenario_id)
    # Selectores y balanceadores DENTRO del pipeline: se ajustan solo en training.
    if select=='MI':
        selector=SelectKBest(partial(mutual_info_classif,random_state=SEED), k=10)
    elif select=='RF':
        selector=SelectFromModel(RandomForestClassifier(n_estimators=100,random_state=SEED,n_jobs=1),threshold='median')
    else: selector='passthrough'
    if balance=='SMOTE': sampler=SMOTE(random_state=SEED,k_neighbors=3)
    elif balance=='ADASYN':sampler=ADASYN(random_state=SEED,n_neighbors=3)
    else:sampler='passthrough'
    if algorithm=='SVM':
        model=SVC(C=1.0,kernel='rbf',gamma='scale',probability=False,random_state=SEED,cache_size=512)
    elif algorithm=='Random Forest':
        model=RandomForestClassifier(n_estimators=200,max_depth=12,random_state=SEED,n_jobs=1)
    elif algorithm=='XGBoost':
        model=XGBClassifier(n_estimators=160,max_depth=3,learning_rate=0.1,
          objective='binary:logistic',eval_metric='logloss',tree_method='hist',
          random_state=SEED,n_jobs=1)
    else: raise ValueError(algorithm)
    return Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),
        ('scaler',StandardScaler()),('selection',selector),('sampler',sampler),('model',model)])


def raw_score(pipe,X):
    if hasattr(pipe,'predict_proba'):
        classes=list(pipe.classes_)
        return np.asarray(pipe.predict_proba(X)[:,classes.index(1)],dtype=float),'probabilidad_no_calibrada'
    if hasattr(pipe,'decision_function'):
        return np.asarray(pipe.decision_function(X),dtype=float),'margen_no_calibrado'
    raise ValueError('No hay score utilizable')


def metrics(pipe,X,y):
    y=np.asarray(y,dtype=int)
    yp=np.asarray(pipe.predict(X),dtype=int)
    s,kind=raw_score(pipe,X)
    if not np.isfinite(s).all(): raise ValueError('Score no finito')
    cm=confusion_matrix(y,yp,labels=[0,1])
    result={'n':len(y),'positivos':int(y.sum()),'prevalencia':float(y.mean()),
       'f1':float(f1_score(y,yp,zero_division=0)),
       'average_precision':float(average_precision_score(y,s)),
       'auc_roc':float(roc_auc_score(y,s)) if len(np.unique(y))==2 else None,
       'precision':float(precision_score(y,yp,zero_division=0)),
       'recall':float(recall_score(y,yp,zero_division=0)),
       'accuracy':float(accuracy_score(y,yp)),
       'tn':int(cm[0,0]),'fp':int(cm[0,1]),'fn':int(cm[1,0]),'tp':int(cm[1,1]),
       'score_tipo':kind}
    return result,s,yp


def ranking_metrics(y,score,loc,fracs=(.1,.2,.3)):
    y=np.asarray(y,dtype=int);n=len(y);pos=int(y.sum())
    if pos==0: return []
    idx=np.arange(n)
    out=[]
    ranks={'ML':np.lexsort((idx,-score)),'LOC':np.lexsort((idx,-np.asarray(loc,dtype=float)))}
    for f in fracs:
        k=max(1,int(np.ceil(n*f)))
        for name,order in ranks.items():
            true=int(y[order[:k]].sum())
            out.append({'metodo':name,'fraccion':f,'k':k,'detectados':true,
               'precision_at_k':true/k,'recall_at_k':true/pos,
               'lift_at_k':(true/k)/float(y.mean())})
        out.append({'metodo':'Aleatorio_esperado','fraccion':f,'k':k,'detectados':None,
            'precision_at_k':float(y.mean()),'recall_at_k':k/n,'lift_at_k':1.0})
    return out



def deduplicar_global(data, seed=SEED):
    """Desduplicación PREVIA a los folds, basada SOLO en vectores X.

    Una huella idéntica en cualquier proyecto puede pertenecer a un solo registro
    del conjunto científico. Se elige de manera determinista un registro mediante
    sha256(huella, proyecto), sin usar las etiquetas ni los resultados de test.
    Todas las eliminaciones se registran. Conservar un único registro no demuestra
    independencia de los proyectos; se considera una sensibilidad metodológica.
    """
    grupos = {}
    for p,(X,y) in data.items():
        for i,h in enumerate(pd.util.hash_pandas_object(X[list(FEATURES)],index=False).to_numpy(dtype=np.uint64)):
            grupos.setdefault(int(h),[]).append((p,i,int(y.iloc[i])))
    conservados={p:set() for p in data}
    con_conflicto=0; filas_con_conflicto=0; cross_groups=0; grupos_repetidos=0
    detalle=[]
    for h,filas in grupos.items():
        if len(filas)>1:
            grupos_repetidos+=1
            if len({p for p,_,_ in filas})>1: cross_groups+=1
            if len({y for _,_,y in filas})>1:
                con_conflicto+=1;filas_con_conflicto+=len(filas)
        # Elección no basada en y ni en orden de proyectos; desempate por posición.
        elegido=min(filas,key=lambda r:(hashlib.sha256(f'{seed}|{h}|{r[0]}'.encode()).hexdigest(),r[1]))
        conservados[elegido[0]].add(elegido[1])
    salida={}; reporte=[]
    for p,(X,y) in data.items():
        indices=sorted(conservados[p])
        X2=X.iloc[indices].reset_index(drop=True)
        y2=y.iloc[indices].reset_index(drop=True)
        if len(X2)<20 or y2.nunique()!=2:
            raise ValueError(f'La deduplicación deja {p} sin muestra/clases adecuadas: n={len(X2)}, clases={y2.value_counts().to_dict()}')
        salida[p]=(X2,y2)
        reporte.append({'proyecto':p,'filas_originales':len(X), 'filas_deduplicadas':len(X2),
          'filas_eliminadas':len(X)-len(X2),'positivos_originales':int(y.sum()),
          'positivos_deduplicados':int(y2.sum()),'prevalencia_deduplicada':float(y2.mean())})
    huellas=[]
    for X,_ in salida.values():
        huellas.extend(pd.util.hash_pandas_object(X[list(FEATURES)],index=False).to_numpy(dtype=np.uint64).tolist())
    assert len(huellas)==len(set(huellas)), 'Error: todavía hay huellas X repetidas'
    resumen={'metodo':'Una ocurrencia por huella de las 21 métricas a escala global ANTES de las particiones; propietario determinado sin y',
       'huellas_unicas':len(grupos),'grupos_repetidos':grupos_repetidos,
       'grupos_repetidos_entre_proyectos':cross_groups,
       'grupos_con_etiquetas_contradictorias':con_conflicto,
       'filas_afectadas_por_etiquetas_contradictorias':filas_con_conflicto,
       'eliminados_totales':sum(len(X) for X,_ in data.values())-sum(len(X) for X,_ in salida.values()),
       'nota':'Los valores de etiqueta contradictorios se auditan, pero NO se usan para decidir qué instancia se conserva; la selección basada en y ocasionaría fuga.'}
    return salida,pd.DataFrame(reporte),resumen


def curva_esfuerzo(y, cost, order):
    """Curva de recall vs proporción de LOC. Área/interpolación lineal por módulo;
    esto NO equivale a inspeccionar literalmente una fracción de un módulo.
    """
    y=np.asarray(y,dtype=int)[order]
    c=np.asarray(cost,dtype=float)[order]
    xs=np.r_[0,np.cumsum(c)/c.sum()]
    ys=np.r_[0,np.cumsum(y)/y.sum()]
    return xs,ys


def area_hasta(xs,ys,fin=.2):
    if fin<=0 or fin>1:raise ValueError(fin)
    mask=(xs<fin)
    x=np.r_[xs[mask],fin]
    y=np.r_[ys[mask],np.interp(fin,xs,ys)]
    return float(np.trapz(y,x) if not hasattr(np,'trapezoid') else np.trapezoid(y,x))


def esfuerzo_qa(y, score, loc, proyecto, contexto, fracciones=(.1,.2,.3)):
    """Popt@20% LOC y Recall@%LOC, sobre MODULOS positivos (no total de bugs).
    Ranking LOC-desc = regla sencilla. OPT/WORST usan y solo como techo de referencia
    para Popt, JAMÁS para entrenar o desplegar el modelo.
    """
    y=np.asarray(y,dtype=int)
    s=np.asarray(score,dtype=float)
    costs=np.asarray(loc,dtype=float)
    if len(y)==0 or y.sum()==0 or len(y)!=len(s) or len(y)!=len(costs):
        raise ValueError('La evaluación de esfuerzo requiere módulos etiquetados positivos y tamaños correspondientes')
    if not np.isfinite(costs).all() or (costs<0).any() or not np.isfinite(s).all():
        raise ValueError('LOC/score deben ser finitos; LOC no puede ser negativo. Revisar origen.')
    # Cada módulo debe costar como mínimo 1 LOC para que la curva sea estrictamente creciente.
    costs=np.maximum(costs,1.0)
    ids=np.arange(len(y))
    ordenes={
      'ML':np.lexsort((ids,-s)),
      'LOC_desc':np.lexsort((ids,-costs)),
      'OPT_teorico':np.lexsort((ids,-y/costs)),
      'WORST_teorico':np.lexsort((ids,y/costs))}
    curvas={name:curva_esfuerzo(y,costs,ord_) for name,ord_ in ordenes.items()}
    area_opt=area_hasta(*curvas['OPT_teorico'])
    area_worst=area_hasta(*curvas['WORST_teorico'])
    den=area_opt-area_worst
    registros=[]
    for name,order in ordenes.items():
        x,r=curvas[name]; area=area_hasta(x,r)
        valores={}
        for f in fracciones:
            # interpolación para la curva integrada; estimación explícitamente ilustrativa de fracción de módulo
            valores[f'recall_interp_{int(f*100)}pct_loc']=float(np.interp(f,x,r))
            # módulos inspeccionados COMPLETAMENTE sin exceder el % LOC: valor conservador y operativo
            num=int(np.searchsorted(x,f,side='right')-1)
            valores[f'recall_modulos_completos_{int(f*100)}pct_loc']=float(y[order[:num]].sum()/y.sum())
        registros.append({'proyecto_test':proyecto,'contexto':contexto,'metodo':name,
          'positivos_totales':int(y.sum()),'modulos':len(y),'loc_total_proxy':float(costs.sum()),
          'area_0_20':area,'popt_20':float((area-area_worst)/den) if den>1e-12 else None,
          **valores})
    return registros


def firma_datos(data):
    h=hashlib.sha256()
    for name in sorted(data):
        X,y=data[name]
        h.update(name.encode())
        h.update(pd.util.hash_pandas_object(X,index=False).to_numpy(dtype=np.uint64).tobytes())
        h.update(np.asarray(y,dtype=np.int8).tobytes())
    return h.hexdigest()


def evaluar_config_fija(data, scenario, algorithm, modo, contexto):
    """LOPO con configuración FIJA, sin selección interna. No es test nuevo del joblib.
    Tras seleccionar una receta con estos proyectos es un análisis descriptivo, NO prueba inédita.
    """
    rows=[]; qa=[]
    for target in data:
        fuentes=[p for p in data if p!=target]
        Xtr,ytr=pack(data,fuentes)
        Xte,yte=data[target]
        pipe=make_pipeline(scenario,algorithm)
        pipe.fit(Xtr,ytr)
        m,score,_=metrics(pipe,Xte,yte)
        rows.append({'modo':modo,'configuracion':contexto,'algoritmo':algorithm,
          'escenario':scenario,'proyecto_test':target,**m})
        qa.extend(esfuerzo_qa(yte,score,Xte['loc'].to_numpy(),target,contexto))
    return rows,qa


def config_hash(audits,projects,data,modo):
    info={'version':EXPERIMENT_VERSION,'datasets':{a['proyecto']:a['sha256'] for a in audits},
       'proyectos':list(projects),'candidatos':list(CANDIDATES),'seed':SEED,
       'modo':modo,'huella_datos_procesados':firma_datos(data),
       'librerias':{lib:version(lib) for lib in ['scikit-learn','imbalanced-learn','xgboost']}}
    return hashlib.sha256(json.dumps(info,sort_keys=True).encode()).hexdigest(),info


def eval_one_candidate(data,train_projects,scenario,algorithm):
    """Validación interna LOGO en PROYECTOS fuente; sin observar proyecto objetivo."""
    folds=[];tic=time.time()
    for validation_project in train_projects:
        fit_projects=[p for p in train_projects if p!=validation_project]
        X_tr,y_tr=pack(data,fit_projects)
        X_val,y_val=data[validation_project]
        pipe=make_pipeline(scenario,algorithm)
        pipe.fit(X_tr,y_tr)
        mm,_,_=metrics(pipe,X_val,y_val)
        folds.append({'proyecto_validacion':validation_project,
            **{k:mm[k] for k in ['n','positivos','f1','average_precision','auc_roc','precision','recall']}})
    return {'escenario':scenario,'algoritmo':algorithm,'n_folds':len(folds),
      'f1_macro':float(np.mean([d['f1'] for d in folds])),
      'ap_macro':float(np.mean([d['average_precision'] for d in folds])),
      'auc_macro':float(np.mean([d['auc_roc'] for d in folds])),
      'segundos':round(time.time()-tic,2),'error':None,'folds':folds}


def select_candidate(rows,expected_folds):
    good=[r for r in rows if r.get('error') is None and r.get('n_folds')==expected_folds]
    if not good:raise RuntimeError('Ningún candidato completo y válido')
    # Criterios fijados antes de ver proyectos reservados: F1 -> AP -> AUC.
    return sorted(good,key=lambda r:(-r['f1_macro'],-r['ap_macro'],-r['auc_macro'],
                                     r['escenario'],r['algoritmo']))[0]


def inner_search(data,projects,checkpoint,lock,*,candidate_limit=None):
    """Checkpoints por candidato (4 o 5 ajustes). Los errores se registran."""
    checkpoint=Path(checkpoint)
    if checkpoint.exists():
        saved=json.loads(checkpoint.read_text(encoding='utf-8'))
        if saved['config_hash']!=lock or saved['proyectos']!=list(projects):
            raise RuntimeError(f'Checkpoint incompatible: {checkpoint}; no se mezclan experimentos')
        rows=saved['rows']
    else: rows=[]
    done={(r['escenario'],r['algoritmo']) for r in rows}
    candidates=CANDIDATES[:candidate_limit] if candidate_limit else CANDIDATES
    for count,(scenario,algorithm) in enumerate(candidates,1):
        if (scenario,algorithm) in done:continue
        print(f'  candidato {count:02}/{len(candidates)} · {algorithm} / {scenario}',flush=True)
        try:
            row=eval_one_candidate(data,projects,scenario,algorithm)
            print(f"      F1(macro)={row['f1_macro']:.3f} · AP={row['ap_macro']:.3f}",flush=True)
        except Exception as exc:
            row={'escenario':scenario,'algoritmo':algorithm,'n_folds':0,
                 'f1_macro':None,'ap_macro':None,'auc_macro':None,
                 'segundos':None,'error':f'{type(exc).__name__}: {str(exc)[:400]}','folds':[]}
            print('      ERROR:',row['error'],flush=True)
        rows.append(row)
        jdump(checkpoint,{'config_hash':lock,'proyectos':list(projects),'rows':rows})
    return rows


def outer_test(data,target,selection,modo):
    sources=[p for p in data if p!=target]
    Xt,yt=pack(data,sources)
    Xo,yo=data[target]
    pipe=make_pipeline(selection['escenario'],selection['algoritmo'])
    pipe.fit(Xt,yt)
    m,score,pred=metrics(pipe,Xo,yo)
    src_hash=set(pd.util.hash_pandas_object(Xt,index=False).to_numpy(dtype=np.uint64).tolist())
    tgt_hash=pd.util.hash_pandas_object(Xo,index=False).to_numpy(dtype=np.uint64)
    unseen=np.asarray([int(h) not in src_hash for h in tgt_hash],dtype=bool)
    m.update({'proyecto_test':target,'proyectos_entrenamiento':sources,
       'escenario':selection['escenario'],'algoritmo':selection['algoritmo'],
       'cv_f1_macro_fuente':selection['f1_macro'],
       'vectores_test_presentes_en_train':int((~unseen).sum()),
       'modo':modo,'validation_protocolo':'nested_leave_one_project_out'})
    ranks=ranking_metrics(yo,score,Xo['loc'].to_numpy())
    for r in ranks:
        r['proyecto_test']=target
        r['modo']=modo
    qa=esfuerzo_qa(yo,score,Xo['loc'].to_numpy(),target,'seleccion_anidada')
    return m,ranks,qa


def run_nested(data,audits,results,*,modo,candidate_limit=None):
    """Evaluación anidada de la REGLA de selección, independiente del modelo final.
    Luego selecciona una configuración mediante LOPO global (cinco proyectos) y
    exporta exactamente UN pipeline entrenado en las fuentes de ese modo.
    """
    results=Path(results);results.mkdir(parents=True,exist_ok=True)
    names=list(data)
    lock,config=config_hash(audits,names,data,modo)
    jdump(results/'configuracion.json',{'hash':lock,'especificacion':config,
      'advertencias':['Procedencia documental no acreditada','LOC aproximación a esfuerzo real',
        'F1 de procedimiento outer NO es una prueba independiente del .joblib final',
        'Popt@20 es área normalizada con interpolación por módulo; reportar junto a resultados de módulos completos']})
    outer=[];rank=[];effort=[];por_algoritmo=[]
    for i,target in enumerate(names,1):
        print(f'\n[{modo}] TEST EXTERNO {i}/{len(names)}: {target}',flush=True)
        source=[p for p in names if p!=target]
        rows=inner_search(data,source,results/f'inner_{target}.json',lock,candidate_limit=candidate_limit)
        chosen=select_candidate(rows,len(source))
        print(f"   Configuración elegida internamente: {chosen['algoritmo']} / {chosen['escenario']}",flush=True)
        for algoritmo in ALGORITHMS:
            candidates_algo=[r for r in rows if r['algoritmo']==algoritmo and r.get('error') is None]
            if not candidates_algo: continue
            elegido_algo=select_candidate(candidates_algo,len(source))
            m,r,e=outer_test(data,target,elegido_algo,modo)
            m['elegido_procedimiento_general']=bool(algoritmo==chosen['algoritmo'])
            por_algoritmo.append(m)
            if algoritmo==chosen['algoritmo']:
                outer.append(m);rank.extend(r);effort.extend(e)
                jdump(results/f'test_externo_{target}.json',m)
            print(f"   {algoritmo}: F1={m['f1']:.3f} | AP={m['average_precision']:.3f}",flush=True)
        pd.DataFrame([{k:v for k,v in r.items() if k!='folds'} for r in rows]).to_csv(
            results/f'candidatos_inner_{target}.csv',index=False)
        pd.DataFrame(outer).to_csv(results/'resultados_por_proyecto.csv',index=False)
        pd.DataFrame(effort).to_csv(results/'qa_esfuerzo_nested.csv',index=False)
    assert len(outer)==len(names),'No todos los folds externos terminaron correctamente'
    pd.DataFrame(por_algoritmo).to_csv(results/'comparacion_algoritmos_outer.csv',index=False)
    comp=pd.DataFrame(por_algoritmo).groupby('algoritmo',as_index=False).agg(
        f1_macro_externo=('f1','mean'),ap_macro_externo=('average_precision','mean'),
        auc_macro_externo=('auc_roc','mean'),recall_macro_externo=('recall','mean'),
        proyectos_evaluados=('proyecto_test','nunique'))
    comp.sort_values(['f1_macro_externo','ap_macro_externo'],ascending=False).to_csv(
        results/'resumen_algoritmos_externos.csv',index=False)
    pd.DataFrame(rank).to_csv(results/'qa_recall_por_modulos.csv',index=False)
    jdump(results/'resultados_outer.json',outer)
    print(f'\n[{modo}] SELECCION GLOBAL mediante cinco folds LOGO',flush=True)
    rows=inner_search(data,names,results/'seleccion_final_cv.json',lock,candidate_limit=candidate_limit)
    winning=select_candidate(rows,len(names))
    pd.DataFrame([{k:v for k,v in row.items() if k!='folds'} for row in rows]).to_csv(
        results/'comparacion_global_cpdp.csv',index=False)
    X_all,y_all=pack(data,names)
    pipe=make_pipeline(winning['escenario'],winning['algoritmo'])
    pipe.fit(X_all,y_all)
    manifest={'protocolo':EXPERIMENT_VERSION,'nombre':'Modelo_general_CPDP',
        'algoritmo':winning['algoritmo'],'escenario':winning['escenario'],
        'seleccion_criterios':'F1 positivo promedio por proyecto; desempate AP y AUC en LOGO',
        'cv_global_f1':winning['f1_macro'],'cv_global_ap':winning['ap_macro'],
        'cv_global_auc':winning['auc_macro'],'datasets_entrenamiento':names,
        'columnas':list(FEATURES),'etiqueta':'1=defecto_historico_registrado',
        'modo_preprocesamiento':modo,'huella_datos_entrenados':firma_datos(data),
        'datos_entrenamiento_cantidad':len(y_all),'documentally_verified':False,
        'score_aviso':'SVM=margin no calibrado / RF y XGB=salida no calibrada',
        'fuentes':{a['proyecto']:a['sha256'] for a in audits},
        'seleccion_global_reentrenada_con_5_proyectos':True,
        'limite':'Sin sexto proyecto externo, F1 global no es prueba independiente del archivo joblib final.',
        'creado_utc':datetime.now(timezone.utc).isoformat()}
    bundle={'pipeline':pipe,'manifest':manifest,'columnas':list(FEATURES)}
    joblib.dump(bundle,results/'modelo_general_cpdp.joblib',compress=3)
    jdump(results/'manifiesto_modelo_general.json',manifest)
    pd.DataFrame(columns=['id_modulo',*FEATURES]).to_csv(results/'plantilla_para_streamlit.csv',index=False)
    # Este contraste de recetas fijas se declara exploratorio cuando ya se escogió con estos mismos proyectos.
    comparacion_fija=[];comparacion_esfuerzo=[]
    for sid,algo,nombre in [('E0_BASELINE','SVM','SVM_base'),
                            ('E7_SMOTE_RF','SVM','SVM_SMOTE_RF')]:
        print(f'[{modo}] Comparación intra-SVM: {nombre}',flush=True)
        rr,qq=evaluar_config_fija(data,sid,algo,modo,nombre)
        comparacion_fija.extend(rr);comparacion_esfuerzo.extend(qq)
    pd.DataFrame(comparacion_fija).to_csv(results/'comparacion_SVM_base_vs_SVM_E7.csv',index=False)
    pd.DataFrame(comparacion_esfuerzo).to_csv(results/'qa_esfuerzo_comparacion_SVM.csv',index=False)
    # Verificaciones reproducibles de congruencia del artefacto.
    loaded=joblib.load(results/'modelo_general_cpdp.joblib')
    assert loaded['manifest']['algoritmo']==winning['algoritmo']
    assert loaded['manifest']['escenario']==winning['escenario']
    assert list(loaded['columnas'])==list(FEATURES)
    assert len(loaded['pipeline'].predict(X_all.head(3)))==3
    summary={'modo':modo,'candidatos_por_busqueda':len(CANDIDATES) if candidate_limit is None else candidate_limit,
       'folds_externos':len(names),'macro_f1_outer':float(np.mean([r['f1'] for r in outer])),
       'macro_ap_outer':float(np.mean([r['average_precision'] for r in outer])),
       'macro_auc_outer':float(np.mean([r['auc_roc'] for r in outer])),
       'ganador_final':{'algoritmo':winning['algoritmo'],'escenario':winning['escenario'],
         'f1_cv_proyectos':winning['f1_macro'],'ap_cv_proyectos':winning['ap_macro']},
       'nota':'Nested outer estima procedimiento selección por proyecto; final es receta global reentrenada. Sin prueba inédita del joblib.'}
    jdump(results/'resumen_final.json',summary)
    return summary


def inference(bundle,df):
    """Solo para prueba de integración; interfaz productiva será Streamlit."""
    if 'id_modulo' in df.columns:
        ids=df['id_modulo'].astype(str).tolist()
    else:ids=[f'modulo_{i+1:04}' for i in range(len(df))]
    X,aliases=harmonize_features(df.drop(columns=['id_modulo'],errors='ignore'), 'CSV externo',allow_extra=False)
    pipe=bundle['pipeline']
    score,kind=raw_score(pipe,X)
    yp=pipe.predict(X)
    out=pd.DataFrame({'id_modulo':ids,'clase_predicha':yp.astype(int),
        'score_riesgo':score,'tipo_score':kind})
    return out.sort_values('score_riesgo',ascending=False,kind='stable').reset_index(drop=True).assign(
        prioridad=lambda x:np.arange(1,len(x)+1))


def descargar_datos_nuevos(raw_dir, timeout=75):
    """Siempre descarga fuentes públicas; no lee carpetas ni resultados de versiones anteriores."""
    import urllib.request
    raw_dir=Path(raw_dir);raw_dir.mkdir(parents=True,exist_ok=True)
    registros=[]
    for p in PROJECTS:
        url=f'{SOURCE_BASE}/{p.lower()}.csv'
        destino=raw_dir/f'{p}.csv'
        if destino.exists():
            raise FileExistsError(f'Para asegurar el inicio desde cero, la ruta ya existe: {destino}')
        request=urllib.request.Request(url,headers={'User-Agent':'tesis-cpdp-reproducible/1.0'})
        with urllib.request.urlopen(request,timeout=timeout) as response:
            blob=response.read(25_000_001)
        if not (1000<len(blob)<=25_000_000):
            raise RuntimeError(f'{p}: tamaño de descarga sospechoso ({len(blob)})')
        destino.write_bytes(blob)
        try:
            _,_,aud=load_project(destino,p)
        except Exception:
            destino.unlink(missing_ok=True)
            raise
        registros.append({'proyecto':p,'url':url,'archivo':destino.name,'sha256':aud['sha256'],
                          'registros':aud['filas'],'variable_objetivo':aud['etiqueta_original']})
        print(f'{p}: {aud["filas"]} registros; SHA-256={aud["sha256"][:12]}...')
    return pd.DataFrame(registros)


## 3. Auditoría, métricas de esfuerzo y líneas base

El módulo escrito en la siguiente celda incluye el análisis de cohortes, el ranking y las métricas de QA. **ManualUp** prioriza módulos por LOC ascendente. **LOC-desc** inspecciona primero los módulos grandes. El ranking ML ordinario usa el margen del SVM sin calibrar; ese margen **no es una probabilidad**.

Se diferencian dos presupuestos: inspeccionar una proporción de **módulos** y revisar una proporción del **LOC total**. En la medición operativa por LOC solo se cuentan módulos completos sin superar el presupuesto. Para Popt se emplea el área de una curva interpolada, que debe interpretarse por separado.

In [ ]:
%%writefile cpdp_evaluacion.py
"""T2 CPDP. Código integrado de auditoría y evaluación del experimento; no requiere otros notebooks.

No reemplazar el artefacto de Streamlit hasta revisión humana y pruebas de integración.
"""
from __future__ import annotations
import hashlib
import json
import os
import sys
import time
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
import joblib
import cpdp_core as base

EXPERIMENTO_ID = 'CPDP_T2_QA_CALIBRACION_ESFUERZO'
METODOS = ('ML', 'ManualUp', 'LOC_desc', 'Aleatorio_esperado')
PRESUPUESTOS = (0.10, 0.20, 0.30)
SEED = 42


def json_save(dest, obj):
    base.jdump(dest, obj)


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def project_keys(data):
    """Huella de las 21 características; no incluye etiqueta ni origen."""
    return {p: pd.util.hash_pandas_object(X.loc[:, list(base.FEATURES)],index=False).to_numpy(dtype=np.uint64)
            for p,(X,_) in data.items()}


def contradiction_audit(data, out_dir):
    """Busca X idénticas con etiquetas discordantes ANTES de particionar.

    La eliminación de grupos basada en y es un análisis de sensibilidad de un
    benchmark histórico completamente etiquetado; no es filtrado aplicable a
    proyectos nuevos sin etiquetas y no se presenta como prueba ciega.
    """
    keys=project_keys(data)
    group={}
    for project,(X,y) in data.items():
        for ix,(h,lab) in enumerate(zip(keys[project],np.asarray(y,dtype=int))):
            group.setdefault(int(h),[]).append((project,int(ix),int(lab)))
    ambiguous={k for k,rows in group.items() if len(set(v[2] for v in rows))>1}
    details=[]
    for h in sorted(ambiguous):
        vals=group[h]
        details.append({'huella_vector_u64':str(h),'filas':len(vals),
                        'proyectos':','.join(sorted(set(v[0] for v in vals))),
                        'conteo_clase_0':sum(v[2]==0 for v in vals),
                        'conteo_clase_1':sum(v[2]==1 for v in vals),
                        'filas_referenciadas':';'.join(f'{p}:{i}' for p,i,_ in vals)})
    out_dir=Path(out_dir);out_dir.mkdir(parents=True,exist_ok=True)
    pd.DataFrame(details).to_csv(out_dir/'grupos_etiquetas_contradictorias.csv',index=False)
    data_filtered={};report=[]
    for p,(X,y) in data.items():
        keep=~np.isin(keys[p],np.array(list(ambiguous),dtype=np.uint64))
        x=X.iloc[np.flatnonzero(keep)].reset_index(drop=True)
        yy=y.iloc[np.flatnonzero(keep)].reset_index(drop=True)
        if len(x)<20 or yy.nunique()!=2:
            raise ValueError(f'{p}: política DESCARTAR_CONFLICTOS no mantiene ambas clases')
        data_filtered[p]=(x,yy)
        report.append({'proyecto':p,'filas_antes':len(y),
                       'filas_eliminadas_por_conflicto':int((~keep).sum()),
                       'filas_despues':len(yy),'positivos_despues':int(yy.sum())})
    pd.DataFrame(report).to_csv(out_dir/'impacto_conflictos_por_proyecto.csv',index=False)
    return data_filtered,{'grupos_contradictorios':len(ambiguous),
                          'filas_en_grupos_contradictorios':sum(len(group[k]) for k in ambiguous),
                          'regla_principal':'Conservar una sola X global, propietario elegido sin consultar y; política heredada y evaluada aquí.',
                          'regla_sensibilidad':'Descartar TODAS las filas cuya X tiene etiquetas contradictorias en el corpus completo; luego deduplicar X sin mirar y.',
                          'advertencia':'El descarte usa etiquetas del corpus entero como limpieza de benchmark; NO puede simularse en un nuevo proyecto sin y. Resultados solo análisis de sensibilidad.'}


def build_cohorts(original,out_dir):
    out_dir=Path(out_dir)
    primary, r_primary, s_primary=base.deduplicar_global(original)
    reduced,s_conf=contradiction_audit(original,out_dir/'auditoria')
    sensitivity,r_sens,s_sens=base.deduplicar_global(reduced)
    for name,data in [('principal_deduplicada',primary),('eliminar_grupos_conflictivos',sensitivity)]:
        audit=base.overlap_audit(data)
        if audit['vectores_identicos'].sum()!=0:
            raise AssertionError(f'{name} todavía tiene solapamiento')
        audit.to_csv(out_dir/'auditoria'/f'solapamiento_{name}.csv',index=False)
    r_primary.to_csv(out_dir/'auditoria'/'dedup_principal_por_proyecto.csv',index=False)
    r_sens.to_csv(out_dir/'auditoria'/'dedup_sensibilidad_por_proyecto.csv',index=False)
    report={'original_total':sum(len(y) for _,y in original.values()),
            'principal_total':sum(len(y) for _,y in primary.values()),
            'sensibilidad_total':sum(len(y) for _,y in sensitivity.values()),
            'principal_resumen':s_primary,'sensibilidad_resumen':s_sens,
            'conflictos':s_conf,
            'huella_principal':base.firma_datos(primary),
            'huella_sensibilidad':base.firma_datos(sensitivity)}
    json_save(out_dir/'auditoria'/'resumen_cohortes.json',report)
    return {'sin_deduplicar':original,'principal_deduplicada':primary,
            'eliminar_grupos_conflictivos':sensitivity},report


def safe_areas(y,c,orders, fraction):
    """Popt normalizado: (A_metodo-A_worst)/(A_opt-A_worst).
    A hasta fracción de LOC con interpolación LINEAL para curva, no inspección
    real de una fracción de módulo; además se reporta recall de módulos completos.
    """
    lines={key:base.curva_esfuerzo(y,c,order) for key,order in orders.items()}
    a_opt=base.area_hasta(*lines['OPT_teorico'],fin=fraction)
    a_worst=base.area_hasta(*lines['WORST_teorico'],fin=fraction)
    return lines, a_opt, a_worst


def ranking_evaluation(y, s, loc, *, project, cohort, procedimiento, random_repeats=50):
    y=np.asarray(y,dtype=int);s=np.asarray(s,dtype=float);loc=np.asarray(loc,dtype=float)
    if len(y)!=len(s) or len(y)!=len(loc) or not np.isfinite(s).all():
        raise ValueError('Score o tamaños incompatibles')
    if not np.isfinite(loc).all() or (loc < 0).any():
        raise ValueError('LOC no finito o negativo; no es esfuerzo válido')
    if int(y.sum())==0 or len(np.unique(y))!=2:
        raise ValueError('Proyecto test sin ambas clases')
    if random_repeats<1: raise ValueError('random_repeats debe ser positivo')
    loc_cost=np.maximum(loc,1.0)
    n=len(y); idx=np.arange(n);positive=int(y.sum())
    orders={
       'ML':np.lexsort((idx,-s)),
       'ManualUp':np.lexsort((idx,loc_cost)),
       'LOC_desc':np.lexsort((idx,-loc_cost)),
       'OPT_teorico':np.lexsort((idx,-y/loc_cost)),
       'WORST_teorico':np.lexsort((idx,y/loc_cost)),
    }
    # La definición de aleatorio como media de permutaciones, no un ranking elegido por etiquetas.
    rng=np.random.default_rng(SEED+sum(map(ord,project))+len(cohort))
    random_orders=[rng.permutation(n) for _ in range(random_repeats)]
    output=[]
    for method in METODOS:
        these_orders=random_orders if method=='Aleatorio_esperado' else [orders[method]]
        for f in PRESUPUESTOS:
            k=max(1,int(np.ceil(n*f)))
            module_recalls=[float(y[order[:k]].sum()/positive) for order in these_orders]
            # Para LOC no se sobrepasa el presupuesto: solo módulos completos.
            budget=f*float(loc_cost.sum())
            values=[]; reals=[]; areas=[]; areas_full=[]
            for order in these_orders:
                xs,ys=base.curva_esfuerzo(y,loc_cost,order)
                cost_used=np.cumsum(loc_cost[order])
                complete=int(np.searchsorted(cost_used,budget,side='right'))
                values.append(float(y[order[:complete]].sum()/positive))
                reals.append(float(np.interp(f,xs,ys)))
                areas.append(base.area_hasta(xs,ys,fin=.2))
                areas_full.append(base.area_hasta(xs,ys,fin=1.0))
            _,opt20,worst20=safe_areas(y,loc_cost,orders,.2)
            _,optfull,worstfull=safe_areas(y,loc_cost,orders,1.0)
            area20=float(np.mean(areas));area_full=float(np.mean(areas_full))
            p20=(area20-worst20)/(opt20-worst20) if opt20-worst20>1e-12 else np.nan
            pfull=(area_full-worstfull)/(optfull-worstfull) if optfull-worstfull>1e-12 else np.nan
            output.append({'cohorte':cohort,'procedimiento':procedimiento,'proyecto':project,
                           'metodo':method,'presupuesto_fraccion':f,
                           'n_modulos':n,'n_positivos':positive,'k_modulos':k,
                           'recall_por_modulos':float(np.mean(module_recalls)),
                           'recall_por_LOC_modulos_completos':float(np.mean(values)),
                           'recall_por_LOC_interpolado':float(np.mean(reals)),
                           'popt_normalizado_0_20':float(p20),
                           'popt_normalizado_0_100':float(pfull),
                           'A20_metodo':area20,'A20_optimo':opt20,'A20_peor':worst20,
                           'A100_metodo':area_full,'A100_optimo':optfull,'A100_peor':worstfull,
                           'loc_total_proxy':float(loc_cost.sum()),
                           'aleatorio_n_permutaciones':random_repeats if method=='Aleatorio_esperado' else 0})
    return output


def evaluate_fixed_configs(data,cohort,result_dir, configs=('E6_ADASYN_MI','E7_SMOTE_RF')):
    result_dir=Path(result_dir);result_dir.mkdir(parents=True,exist_ok=True)
    rows=[]; rank=[]
    for target in base.PROJECTS:
        src=[p for p in base.PROJECTS if p!=target]
        Xt,yt=base.pack(data,src)
        Xo,yo=data[target]
        for scenario in configs:
            print(f'[{cohort}] prueba LOPO fija · {target} · {scenario}',flush=True)
            pipe=base.make_pipeline(scenario,'SVM')
            start=time.perf_counter()
            pipe.fit(Xt,yt)
            t_fit=time.perf_counter()-start
            start=time.perf_counter()
            m,score,pred=base.metrics(pipe,Xo,yo)
            t_test=time.perf_counter()-start
            rows.append({'cohorte':cohort,'proyecto':target,'algoritmo':'SVM',
                         'escenario':scenario,'segundos_entrenamiento':t_fit,
                         'segundos_inferencia_evaluacion':t_test,**m})
            rank.extend(ranking_evaluation(yo,score,Xo['loc'],project=target,
                          cohort=cohort,procedimiento=f'SVM_{scenario}'))
    rr=pd.DataFrame(rows);qq=pd.DataFrame(rank)
    rr.to_csv(result_dir/'E6_vs_E7_por_proyecto.csv',index=False)
    qq.to_csv(result_dir/'E6_vs_E7_ranking_por_proyecto.csv',index=False)
    return rr,qq


def evaluate_reference_nested(data,cohort,configs_by_project,result_dir):
    """NO reoptimiza las 27 recetas: reutiliza selecciones anidadas de un experimento anterior
    únicamente si hashes de CSV y firma de la cohorte coinciden con las huellas incorporadas.
    Reentrena la receta preseleccionada SIN tocar etiquetas del proyecto test.
    """
    rows=[]; ranks=[]
    for p in base.PROJECTS:
        scenario=configs_by_project[p]['escenario'];algo=configs_by_project[p]['algoritmo']
        src=[q for q in base.PROJECTS if q!=p]
        Xtr,ytr=base.pack(data,src); Xte,yte=data[p]
        print(f'[Selección previa {cohort}] TEST {p}: {algo} / {scenario}',flush=True)
        pipe=base.make_pipeline(scenario,algo)
        pipe.fit(Xtr,ytr)
        m,score,_=base.metrics(pipe,Xte,yte)
        rows.append({'cohorte':cohort,'proyecto':p,'algoritmo':algo,'escenario':scenario,**m})
        ranks.extend(ranking_evaluation(yte,score,Xte['loc'],project=p,cohort=cohort,
                                        procedimiento='seleccion_previa_reentrenada'))
    result_dir=Path(result_dir);result_dir.mkdir(parents=True,exist_ok=True)
    rr=pd.DataFrame(rows);qq=pd.DataFrame(ranks)
    rr.to_csv(result_dir/'seleccion_previa_resultados_por_proyecto.csv',index=False)
    qq.to_csv(result_dir/'seleccion_previa_metricas_QA.csv',index=False)
    return rr,qq


def paired_tests(df, cohort, output_dir):
    """Comparación E6/E7 apareada por proyecto; p descriptivos, no confirmatorios.
    Con 5 diferencias sin empates, el menor p exacto bilateral de Wilcoxon es 0.0625.
    """
    subset=df[df['cohorte']==cohort]
    records=[]
    p_values=[]
    for key in ['f1','average_precision','auc_roc','recall']:
        pivot=subset.pivot(index='proyecto',columns='escenario',values=key)
        a=pivot['E6_ADASYN_MI'].to_numpy(dtype=float)
        b=pivot['E7_SMOTE_RF'].to_numpy(dtype=float)
        delta=a-b
        if not np.isfinite(delta).all(): raise ValueError(f'{key}: faltan métricas por proyecto')
        if np.allclose(delta,0):p_value=1.0;stat=0.0
        else:
            res=wilcoxon(delta,zero_method='wilcox',alternative='two-sided',method='exact')
            p_value=float(res.pvalue);stat=float(res.statistic)
        rng=np.random.default_rng(312)
        boots=np.mean(rng.choice(delta,size=(10000,len(delta)),replace=True),axis=1)
        rec={'cohorte':cohort,'metrica':key,'n_proyectos':len(delta),
             'media_E6':float(a.mean()),'media_E7':float(b.mean()),
             'media_diferencia_E6_menos_E7':float(delta.mean()),
             'mediana_diferencia':float(np.median(delta)),
             'desviacion_F1_E6_o_metrica':float(np.std(a,ddof=1)),
             'desviacion_F1_E7_o_metrica':float(np.std(b,ddof=1)),
             'proyectos_E6_mayor':int((delta>0).sum()),'proyectos_E7_mayor':int((delta<0).sum()),
             'wilcoxon_w':stat,'p_bilateral_exacto_exploratorio':p_value,
             'bootstrap_intervalo_95_descriptivo_inferior':float(np.quantile(boots,.025)),
             'bootstrap_intervalo_95_descriptivo_superior':float(np.quantile(boots,.975)),
             'precaucion':'Solo 5 proyectos; sin significancia no equivale a equivalencia; comparaciones en corpus usado para selección.'}
        records.append(rec)
    table=pd.DataFrame(records)
    output_dir=Path(output_dir);output_dir.mkdir(parents=True,exist_ok=True)
    table.to_csv(output_dir/f'prueba_pareada_{cohort}.csv',index=False)
    return table


def summarize_rankings(tables,output_dir):
    rows=pd.concat(tables,ignore_index=True)
    rows.to_csv(Path(output_dir)/'QA_detalle_por_proyecto.csv',index=False)
    fields=['recall_por_modulos','recall_por_LOC_modulos_completos','recall_por_LOC_interpolado',
            'popt_normalizado_0_20','popt_normalizado_0_100']
    summary=(rows.groupby(['cohorte','procedimiento','metodo','presupuesto_fraccion'],as_index=False)
                 [fields].mean())
    summary.to_csv(Path(output_dir)/'QA_promedios_proyecto.csv',index=False)
    return summary


def export_model_candidates(data,root,source_hashes):
    """EXPORTA CANDIDATOS, NO DESPLIEGA. Modelos entrenados con 5 proyectos.
    Estos pipelines no tienen prueba con un sexto proyecto no visto.
    """
    root=Path(root);root.mkdir(parents=True,exist_ok=True)
    X,y=base.pack(data,base.PROJECTS)
    report=[]
    for scen in ['E6_ADASYN_MI','E7_SMOTE_RF']:
        dest=root/scen;dest.mkdir(parents=True,exist_ok=True)
        pipe=base.make_pipeline(scen,'SVM');pipe.fit(X,y)
        manifest={'protocolo':EXPERIMENTO_ID,'nombre':'Modelo_general_CPDP',
                  'algoritmo':'SVM','escenario':scen,'columnas':list(base.FEATURES),
                  'version_modelo':f'{EXPERIMENTO_ID}_{scen}',
                  'dataset_fuentes_sha256':source_hashes,'datos_entrenamiento_cantidad':len(y),
                  'huella_datos_entrenados':base.firma_datos(data),
                  'score_tipo':'margen_svm_no_calibrado',
                  'documentally_verified':False,'candidato_no_desplegado':True,
                  'limite':'Sin sexto proyecto independiente. Requiere verificación Streamlit + P01-P08.',
                  'creado_utc':datetime.now(timezone.utc).isoformat()}
        bundle={'pipeline':pipe,'manifest':manifest,'columnas':list(base.FEATURES)}
        path=dest/'modelo_general_cpdp.joblib'
        joblib.dump(bundle,path,compress=3)
        loaded=joblib.load(path)
        assert loaded['manifest']['escenario']==scen
        assert list(loaded['columnas'])==list(base.FEATURES)
        test=loaded['pipeline'].predict(X.iloc[:5])
        assert len(test)==5
        base.jdump(dest/'manifiesto_modelo_general.json',manifest)
        base.jdump(dest/'integridad_exportacion.json',{'sha256_modelo':base.sha256(path),
                             'sha256_manifiesto':base.sha256(dest/'manifiesto_modelo_general.json'),
                             'no_reemplazar_produccion_sin_aprobacion':True})
        report.append({'escenario':scen,'filas_train':len(y),'sha256':base.sha256(path),
                       'estado':'CANDIDATO_NO_DESPLEGADO'})
    pd.DataFrame(report).to_csv(root/'candidatos_exportados.csv',index=False)
    base.jdump(root/'decision_modelo.json',{
       'estado':'PENDIENTE_REVISAR_CON_ASESOR','configuracion_actual_streamlit':'E6_ADASYN_MI',
       'configuracion_previa_seleccionada':'E6_ADASYN_MI',
       'nota':'Wilcoxon con 5 proyectos tiene poca potencia. No significancia NO demuestra equivalencia. Evitar seleccionar en base al test externo. La decisión requiere regla documentada, análisis de estabilidad/costo y pruebas de integración.'})
    return pd.DataFrame(report)


def quick_selfcheck():
    # Test de las funciones sin entrenar ningún modelo ni consultar fuentes.
    y=np.array([1,0,0,0,0,1,0,0,0,0],dtype=int)
    loc=np.array([1,50,60,70,80,2,90,100,110,120],dtype=float)
    score=np.array([9,0,1,2,3,8,4,5,6,7],dtype=float)
    x=ranking_evaluation(y,score,loc,project='PRUEBA',cohort='PRUEBA',procedimiento='PRUEBA',random_repeats=3)
    assert len(x)==len(METODOS)*len(PRESUPUESTOS)
    for r in x:
        assert 0<=r['recall_por_modulos']<=1
        assert 0<=r['recall_por_LOC_modulos_completos']<=1
        assert 0<=r['recall_por_LOC_interpolado']<=1
        if r['metodo']=='ManualUp' and r['presupuesto_fraccion']==.2:
            assert r['recall_por_LOC_modulos_completos']==1
    # Popt normalizado debe devolver 1 (óptimo) y 0 (peor), en curvas ideales.
    orders={'OPT_teorico':np.lexsort((np.arange(10),-y/loc)),
            'WORST_teorico':np.lexsort((np.arange(10),y/loc))}
    _,opt,worst=safe_areas(y,loc,orders,.2)
    assert opt>worst
    assert abs((opt-worst)/(opt-worst)-1)<1e-12
    print('Pruebas rápidas: OK (ranking, LOC, Popt y presupuestos).')


## 4. Huellas de los datos y recetas de referencia incorporadas

La celda siguiente contiene información de referencia integrada en el propio cuaderno: hashes SHA-256 de los CSV, la huella del conjunto deduplicado y selecciones de recetas por partición calculadas en una investigación anterior. **Estas selecciones no se vuelven a optimizar aquí por defecto**; se reentrenan sobre los datos verificados, separando el proyecto de prueba.

Los resultados de comparación de algoritmos guardados en esa referencia se identifican expresamente como antecedentes, no como mediciones nuevas. No se necesita ningún archivo externo adicional para recuperar dicha información.

In [ ]:
%%writefile referencia_experimental_incluida.json
{
  "identificacion": "Selecciones previas incorporadas para reproducción controlada; no son nuevas pruebas externas.",
  "origen": "Experimento original de selección CPDP T2 (octubre de 2026). Tablas y selecciones conservadas íntegramente en el propio cuaderno.",
  "sha256_fuentes": {
    "CM1": "c05e4834217c7b0af59f4615c4972b36ef636d2f8414141de6e523bb82e70ac5",
    "JM1": "013cb035b07f176e217c0db3abeaeeb214d262e1a63b237a44737ea0f301cc03",
    "KC1": "bdc2221d5490243bd033baaf35a1a906a460cd88a54444dfe8be92aba430e798",
    "KC2": "fedb0f8b4e131dad110e2a7713f7d3b1c0efb68a55b7df7c469610998dafbec7",
    "PC1": "9331ea04c83dfbcb618dbb707c8123e661f2ac5f59c85c03031f46fa0ce9f3e4"
  },
  "hash_proceso_dedup": "a633609a3bb142fe82940c334042c255984e9cebed2216897a06ea94e50c68fd",
  "esquema": {
    "fuente": "Procedimiento de investigación original de octubre de 2026, preservado por huella de datos",
    "librerias": {
      "scikit-learn": "1.6.1",
      "imbalanced-learn": "0.14.2",
      "xgboost": "3.4.1"
    }
  },
  "seleccion_previa_por_fold": {
    "sin_deduplicar": {
      "CM1": {
        "algoritmo": "XGBoost",
        "escenario": "E5_SMOTE_MI"
      },
      "JM1": {
        "algoritmo": "XGBoost",
        "escenario": "E1_SMOTE"
      },
      "KC1": {
        "algoritmo": "SVM",
        "escenario": "E7_SMOTE_RF"
      },
      "KC2": {
        "algoritmo": "XGBoost",
        "escenario": "E5_SMOTE_MI"
      },
      "PC1": {
        "algoritmo": "XGBoost",
        "escenario": "E6_ADASYN_MI"
      }
    },
    "con_deduplicacion_global": {
      "CM1": {
        "algoritmo": "SVM",
        "escenario": "E7_SMOTE_RF"
      },
      "JM1": {
        "algoritmo": "SVM",
        "escenario": "E5_SMOTE_MI"
      },
      "KC1": {
        "algoritmo": "SVM",
        "escenario": "E6_ADASYN_MI"
      },
      "KC2": {
        "algoritmo": "SVM",
        "escenario": "E8_ADASYN_RF"
      },
      "PC1": {
        "algoritmo": "SVM",
        "escenario": "E8_ADASYN_RF"
      }
    }
  },
  "comparacion_global_previa": [
    {
      "escenario": "E0_BASELINE",
      "algoritmo": "SVM",
      "n_folds": "5",
      "f1_macro": "0.14492878761754494",
      "ap_macro": "0.3415796595590125",
      "auc_macro": "0.6311589014761577",
      "segundos": "25.43",
      "error": ""
    },
    {
      "escenario": "E0_BASELINE",
      "algoritmo": "Random Forest",
      "n_folds": "5",
      "f1_macro": "0.18782009640359493",
      "ap_macro": "0.3794736275288426",
      "auc_macro": "0.7286570138739241",
      "segundos": "24.02",
      "error": ""
    },
    {
      "escenario": "E0_BASELINE",
      "algoritmo": "XGBoost",
      "n_folds": "5",
      "f1_macro": "0.2023042978420599",
      "ap_macro": "0.3783821781899256",
      "auc_macro": "0.7186341200700153",
      "segundos": "1.47",
      "error": ""
    },
    {
      "escenario": "E1_SMOTE",
      "algoritmo": "SVM",
      "n_folds": "5",
      "f1_macro": "0.40550020647590285",
      "ap_macro": "0.3375560005709139",
      "auc_macro": "0.7229143825199964",
      "segundos": "78.1",
      "error": ""
    },
    {
      "escenario": "E1_SMOTE",
      "algoritmo": "Random Forest",
      "n_folds": "5",
      "f1_macro": "0.3722983998389717",
      "ap_macro": "0.37087093957278466",
      "auc_macro": "0.7141140794380751",
      "segundos": "47.33",
      "error": ""
    },
    {
      "escenario": "E1_SMOTE",
      "algoritmo": "XGBoost",
      "n_folds": "5",
      "f1_macro": "0.36955413869555964",
      "ap_macro": "0.376282463693547",
      "auc_macro": "0.7069177536880089",
      "segundos": "2.58",
      "error": ""
    },
    {
      "escenario": "E2_ADASYN",
      "algoritmo": "SVM",
      "n_folds": "5",
      "f1_macro": "0.391548319038306",
      "ap_macro": "0.32372966682644655",
      "auc_macro": "0.7154068637292256",
      "segundos": "81.95",
      "error": ""
    },
    {
      "escenario": "E2_ADASYN",
      "algoritmo": "Random Forest",
      "n_folds": "5",
      "f1_macro": "0.36773157503496123",
      "ap_macro": "0.3639072055593452",
      "auc_macro": "0.7066254090356886",
      "segundos": "48.42",
      "error": ""
    },
    {
      "escenario": "E2_ADASYN",
      "algoritmo": "XGBoost",
      "n_folds": "5",
      "f1_macro": "0.3521591412947226",
      "ap_macro": "0.36709700434036313",
      "auc_macro": "0.7025228134256032",
      "segundos": "3.98",
      "error": ""
    },
    {
      "escenario": "E3_MI",
      "algoritmo": "SVM",
      "n_folds": "5",
      "f1_macro": "0.10060944826307658",
      "ap_macro": "0.3541046042651662",
      "auc_macro": "0.6555449478573381",
      "segundos": "26.35",
      "error": ""
    },
    {
      "escenario": "E3_MI",
      "algoritmo": "Random Forest",
      "n_folds": "5",
      "f1_macro": "0.18997514788169503",
      "ap_macro": "0.3584344880046316",
      "auc_macro": "0.7030284180249902",
      "segundos": "26.07",
      "error": ""
    },
    {
      "escenario": "E3_MI",
      "algoritmo": "XGBoost",
      "n_folds": "5",
      "f1_macro": "0.24582287406636044",
      "ap_macro": "0.3853470584988482",
      "auc_macro": "0.7267028169201009",
      "segundos": "5.4",
      "error": ""
    },
    {
      "escenario": "E4_RF_FS",
      "algoritmo": "SVM",
      "n_folds": "5",
      "f1_macro": "0.10531647582030557",
      "ap_macro": "0.34938597514770864",
      "auc_macro": "0.6526626650483139",
      "segundos": "39.34",
      "error": ""
    },
    {
      "escenario": "E4_RF_FS",
      "algoritmo": "Random Forest",
      "n_folds": "5",
      "f1_macro": "0.17100350259946046",
      "ap_macro": "0.3595756582442164",
      "auc_macro": "0.7023816901687193",
      "segundos": "41.54",
      "error": ""
    },
    {
      "escenario": "E4_RF_FS",
      "algoritmo": "XGBoost",
      "n_folds": "5",
      "f1_macro": "0.2426462818438731",
      "ap_macro": "0.4043430595333275",
      "auc_macro": "0.7282563768835841",
      "segundos": "18.14",
      "error": ""
    },
    {
      "escenario": "E5_SMOTE_MI",
      "algoritmo": "SVM",
      "n_folds": "5",
      "f1_macro": "0.4169350885987867",
      "ap_macro": "0.3750417073677589",
      "auc_macro": "0.7294029490861246",
      "segundos": "78.25",
      "error": ""
    },
    {
      "escenario": "E5_SMOTE_MI",
      "algoritmo": "Random Forest",
      "n_folds": "5",
      "f1_macro": "0.37287808770738573",
      "ap_macro": "0.3560415836107226",
      "auc_macro": "0.6858022876486339",
      "segundos": "45.13",
      "error": ""
    },
    {
      "escenario": "E5_SMOTE_MI",
      "algoritmo": "XGBoost",
      "n_folds": "5",
      "f1_macro": "0.41373648307252964",
      "ap_macro": "0.3772680684048374",
      "auc_macro": "0.704591669352949",
      "segundos": "6.93",
      "error": ""
    },
    {
      "escenario": "E6_ADASYN_MI",
      "algoritmo": "SVM",
      "n_folds": "5",
      "f1_macro": "0.41911964964178683",
      "ap_macro": "0.36036126620855324",
      "auc_macro": "0.7268613482845449",
      "segundos": "81.21",
      "error": ""
    },
    {
      "escenario": "E6_ADASYN_MI",
      "algoritmo": "Random Forest",
      "n_folds": "5",
      "f1_macro": "0.3594157230140363",
      "ap_macro": "0.35491564182459384",
      "auc_macro": "0.6731357292626908",
      "segundos": "45.58",
      "error": ""
    },
    {
      "escenario": "E6_ADASYN_MI",
      "algoritmo": "XGBoost",
      "n_folds": "5",
      "f1_macro": "0.403678778137179",
      "ap_macro": "0.3765394508956566",
      "auc_macro": "0.7124914927192008",
      "segundos": "7.24",
      "error": ""
    },
    {
      "escenario": "E7_SMOTE_RF",
      "algoritmo": "SVM",
      "n_folds": "5",
      "f1_macro": "0.4146422654854806",
      "ap_macro": "0.3628130813088917",
      "auc_macro": "0.7259930223723162",
      "segundos": "90.99",
      "error": ""
    },
    {
      "escenario": "E7_SMOTE_RF",
      "algoritmo": "Random Forest",
      "n_folds": "5",
      "f1_macro": "0.3767717554251294",
      "ap_macro": "0.35488255009028463",
      "auc_macro": "0.692881584519877",
      "segundos": "62.62",
      "error": ""
    },
    {
      "escenario": "E7_SMOTE_RF",
      "algoritmo": "XGBoost",
      "n_folds": "5",
      "f1_macro": "0.40520940685681184",
      "ap_macro": "0.3837535761251324",
      "auc_macro": "0.7051439537432108",
      "segundos": "17.82",
      "error": ""
    },
    {
      "escenario": "E8_ADASYN_RF",
      "algoritmo": "SVM",
      "n_folds": "5",
      "f1_macro": "0.41893019996276737",
      "ap_macro": "0.35028813809430986",
      "auc_macro": "0.7271926498887128",
      "segundos": "93.79",
      "error": ""
    },
    {
      "escenario": "E8_ADASYN_RF",
      "algoritmo": "Random Forest",
      "n_folds": "5",
      "f1_macro": "0.3631593385117951",
      "ap_macro": "0.34203764903499695",
      "auc_macro": "0.6850782471744263",
      "segundos": "63.45",
      "error": ""
    },
    {
      "escenario": "E8_ADASYN_RF",
      "algoritmo": "XGBoost",
      "n_folds": "5",
      "f1_macro": "0.39074041011583244",
      "ap_macro": "0.3743571127858981",
      "auc_macro": "0.7113830775276954",
      "segundos": "18.48",
      "error": ""
    }
  ],
  "comparacion_algoritmos_externa_previa": [
    {
      "n": "439",
      "positivos": "46",
      "prevalencia": "0.10478359908883828",
      "f1": "0.3333333333333333",
      "average_precision": "0.25850221543119584",
      "auc_roc": "0.7665670981303242",
      "precision": "0.24489795918367346",
      "recall": "0.5217391304347826",
      "accuracy": "0.7813211845102506",
      "tn": "319",
      "fp": "74",
      "fn": "22",
      "tp": "24",
      "score_tipo": "margen_no_calibrado",
      "proyecto_test": "CM1",
      "proyectos_entrenamiento": "['JM1', 'KC1', 'KC2', 'PC1']",
      "escenario": "E7_SMOTE_RF",
      "algoritmo": "SVM",
      "cv_f1_macro_fuente": "0.43423549610019185",
      "vectores_test_presentes_en_train": "0",
      "modo": "con_deduplicacion_global",
      "validation_protocolo": "nested_leave_one_project_out",
      "elegido_procedimiento_general": "True"
    },
    {
      "n": "439",
      "positivos": "46",
      "prevalencia": "0.10478359908883828",
      "f1": "0.15384615384615385",
      "average_precision": "0.19502431343877547",
      "auc_roc": "0.639119371611904",
      "precision": "0.1875",
      "recall": "0.13043478260869565",
      "accuracy": "0.8496583143507973",
      "tn": "367",
      "fp": "26",
      "fn": "40",
      "tp": "6",
      "score_tipo": "probabilidad_no_calibrada",
      "proyecto_test": "CM1",
      "proyectos_entrenamiento": "['JM1', 'KC1', 'KC2', 'PC1']",
      "escenario": "E6_ADASYN_MI",
      "algoritmo": "Random Forest",
      "cv_f1_macro_fuente": "0.4149908323136226",
      "vectores_test_presentes_en_train": "0",
      "modo": "con_deduplicacion_global",
      "validation_protocolo": "nested_leave_one_project_out",
      "elegido_procedimiento_general": "False"
    },
    {
      "n": "439",
      "positivos": "46",
      "prevalencia": "0.10478359908883828",
      "f1": "0.3484848484848485",
      "average_precision": "0.2582922238134469",
      "auc_roc": "0.7509403695099015",
      "precision": "0.26744186046511625",
      "recall": "0.5",
      "accuracy": "0.8041002277904328",
      "tn": "330",
      "fp": "63",
      "fn": "23",
      "tp": "23",
      "score_tipo": "probabilidad_no_calibrada",
      "proyecto_test": "CM1",
      "proyectos_entrenamiento": "['JM1', 'KC1', 'KC2', 'PC1']",
      "escenario": "E5_SMOTE_MI",
      "algoritmo": "XGBoost",
      "cv_f1_macro_fuente": "0.4198552656961345",
      "vectores_test_presentes_en_train": "0",
      "modo": "con_deduplicacion_global",
      "validation_protocolo": "nested_leave_one_project_out",
      "elegido_procedimiento_general": "False"
    },
    {
      "n": "8787",
      "positivos": "1996",
      "prevalencia": "0.2271537498577444",
      "f1": "0.423562412342216",
      "average_precision": "0.3576886036656904",
      "auc_roc": "0.6582948329290004",
      "precision": "0.325782092772384",
      "recall": "0.6052104208416834",
      "accuracy": "0.6258108569477637",
      "tn": "4291",
      "fp": "2500",
      "fn": "788",
      "tp": "1208",
      "score_tipo": "margen_no_calibrado",
      "proyecto_test": "JM1",
      "proyectos_entrenamiento": "['CM1', 'KC1', 'KC2', 'PC1']",
      "escenario": "E5_SMOTE_MI",
      "algoritmo": "SVM",
      "cv_f1_macro_fuente": "0.38826137304804464",
      "vectores_test_presentes_en_train": "0",
      "modo": "con_deduplicacion_global",
      "validation_protocolo": "nested_leave_one_project_out",
      "elegido_procedimiento_general": "True"
    },
    {
      "n": "8787",
      "positivos": "1996",
      "prevalencia": "0.2271537498577444",
      "f1": "0.3657844990548204",
      "average_precision": "0.3349537097057089",
      "auc_roc": "0.6144855976125422",
      "precision": "0.34615384615384615",
      "recall": "0.3877755511022044",
      "accuracy": "0.6945487652213497",
      "tn": "5329",
      "fp": "1462",
      "fn": "1222",
      "tp": "774",
      "score_tipo": "probabilidad_no_calibrada",
      "proyecto_test": "JM1",
      "proyectos_entrenamiento": "['CM1', 'KC1', 'KC2', 'PC1']",
      "escenario": "E5_SMOTE_MI",
      "algoritmo": "Random Forest",
      "cv_f1_macro_fuente": "0.36425637020726753",
      "vectores_test_presentes_en_train": "0",
      "modo": "con_deduplicacion_global",
      "validation_protocolo": "nested_leave_one_project_out",
      "elegido_procedimiento_general": "False"
    },
    {
      "n": "8787",
      "positivos": "1996",
      "prevalencia": "0.2271537498577444",
      "f1": "0.3832923832923833",
      "average_precision": "0.363189372851233",
      "auc_roc": "0.6381777691740425",
      "precision": "0.34582829504232165",
      "recall": "0.42985971943887774",
      "accuracy": "0.6857858199613065",
      "tn": "5168",
      "fp": "1623",
      "fn": "1138",
      "tp": "858",
      "score_tipo": "probabilidad_no_calibrada",
      "proyecto_test": "JM1",
      "proyectos_entrenamiento": "['CM1', 'KC1', 'KC2', 'PC1']",
      "escenario": "E5_SMOTE_MI",
      "algoritmo": "XGBoost",
      "cv_f1_macro_fuente": "0.37376794939697944",
      "vectores_test_presentes_en_train": "0",
      "modo": "con_deduplicacion_global",
      "validation_protocolo": "nested_leave_one_project_out",
      "elegido_procedimiento_general": "False"
    },
    {
      "n": "1166",
      "positivos": "309",
      "prevalencia": "0.26500857632933106",
      "f1": "0.42201834862385323",
      "average_precision": "0.385212010485827",
      "auc_roc": "0.642151253903698",
      "precision": "0.4",
      "recall": "0.44660194174757284",
      "accuracy": "0.6758147512864494",
      "tn": "650",
      "fp": "207",
      "fn": "171",
      "tp": "138",
      "score_tipo": "margen_no_calibrado",
      "proyecto_test": "KC1",
      "proyectos_entrenamiento": "['CM1', 'JM1', 'KC2', 'PC1']",
      "escenario": "E6_ADASYN_MI",
      "algoritmo": "SVM",
      "cv_f1_macro_fuente": "0.4231832795613573",
      "vectores_test_presentes_en_train": "0",
      "modo": "con_deduplicacion_global",
      "validation_protocolo": "nested_leave_one_project_out",
      "elegido_procedimiento_general": "True"
    },
    {
      "n": "1166",
      "positivos": "309",
      "prevalencia": "0.26500857632933106",
      "f1": "0.35873015873015873",
      "average_precision": "0.3562902300116881",
      "auc_roc": "0.5941664495323115",
      "precision": "0.35202492211838005",
      "recall": "0.3656957928802589",
      "accuracy": "0.6535162950257289",
      "tn": "649",
      "fp": "208",
      "fn": "196",
      "tp": "113",
      "score_tipo": "probabilidad_no_calibrada",
      "proyecto_test": "KC1",
      "proyectos_entrenamiento": "['CM1', 'JM1', 'KC2', 'PC1']",
      "escenario": "E7_SMOTE_RF",
      "algoritmo": "Random Forest",
      "cv_f1_macro_fuente": "0.3622694256080422",
      "vectores_test_presentes_en_train": "0",
      "modo": "con_deduplicacion_global",
      "validation_protocolo": "nested_leave_one_project_out",
      "elegido_procedimiento_general": "False"
    },
    {
      "n": "1166",
      "positivos": "309",
      "prevalencia": "0.26500857632933106",
      "f1": "0.40522875816993464",
      "average_precision": "0.3688919280080596",
      "auc_roc": "0.6170524105689676",
      "precision": "0.40924092409240925",
      "recall": "0.40129449838187703",
      "accuracy": "0.6878216123499142",
      "tn": "678",
      "fp": "179",
      "fn": "185",
      "tp": "124",
      "score_tipo": "probabilidad_no_calibrada",
      "proyecto_test": "KC1",
      "proyectos_entrenamiento": "['CM1', 'JM1', 'KC2', 'PC1']",
      "escenario": "E6_ADASYN_MI",
      "algoritmo": "XGBoost",
      "cv_f1_macro_fuente": "0.41553979345884107",
      "vectores_test_presentes_en_train": "0",
      "modo": "con_deduplicacion_global",
      "validation_protocolo": "nested_leave_one_project_out",
      "elegido_procedimiento_general": "False"
    },
    {
      "n": "351",
      "positivos": "98",
      "prevalencia": "0.2792022792022792",
      "f1": "0.6283524904214559",
      "average_precision": "0.550350917865933",
      "auc_roc": "0.802895861902073",
      "precision": "0.5030674846625767",
      "recall": "0.8367346938775511",
      "accuracy": "0.7236467236467237",
      "tn": "172",
      "fp": "81",
      "fn": "16",
      "tp": "82",
      "score_tipo": "margen_no_calibrado",
      "proyecto_test": "KC2",
      "proyectos_entrenamiento": "['CM1', 'JM1', 'KC1', 'PC1']",
      "escenario": "E8_ADASYN_RF",
      "algoritmo": "SVM",
      "cv_f1_macro_fuente": "0.36767251486920605",
      "vectores_test_presentes_en_train": "0",
      "modo": "con_deduplicacion_global",
      "validation_protocolo": "nested_leave_one_project_out",
      "elegido_procedimiento_general": "True"
    },
    {
      "n": "351",
      "positivos": "98",
      "prevalencia": "0.2792022792022792",
      "f1": "0.6333333333333333",
      "average_precision": "0.6738604310219769",
      "auc_roc": "0.8199161087359844",
      "precision": "0.5352112676056338",
      "recall": "0.7755102040816326",
      "accuracy": "0.7492877492877493",
      "tn": "187",
      "fp": "66",
      "fn": "22",
      "tp": "76",
      "score_tipo": "probabilidad_no_calibrada",
      "proyecto_test": "KC2",
      "proyectos_entrenamiento": "['CM1', 'JM1', 'KC1', 'PC1']",
      "escenario": "E6_ADASYN_MI",
      "algoritmo": "Random Forest",
      "cv_f1_macro_fuente": "0.29134016428306403",
      "vectores_test_presentes_en_train": "0",
      "modo": "con_deduplicacion_global",
      "validation_protocolo": "nested_leave_one_project_out",
      "elegido_procedimiento_general": "False"
    },
    {
      "n": "351",
      "positivos": "98",
      "prevalencia": "0.2792022792022792",
      "f1": "0.5963302752293578",
      "average_precision": "0.6524085244082055",
      "auc_roc": "0.8038638380253287",
      "precision": "0.5416666666666666",
      "recall": "0.6632653061224489",
      "accuracy": "0.7492877492877493",
      "tn": "198",
      "fp": "55",
      "fn": "33",
      "tp": "65",
      "score_tipo": "probabilidad_no_calibrada",
      "proyecto_test": "KC2",
      "proyectos_entrenamiento": "['CM1', 'JM1', 'KC1', 'PC1']",
      "escenario": "E6_ADASYN_MI",
      "algoritmo": "XGBoost",
      "cv_f1_macro_fuente": "0.3442457437862564",
      "vectores_test_presentes_en_train": "0",
      "modo": "con_deduplicacion_global",
      "validation_protocolo": "nested_leave_one_project_out",
      "elegido_procedimiento_general": "False"
    },
    {
      "n": "947",
      "positivos": "70",
      "prevalencia": "0.07391763463569166",
      "f1": "0.25384615384615383",
      "average_precision": "0.19345633980321864",
      "auc_roc": "0.7684557745561166",
      "precision": "0.1736842105263158",
      "recall": "0.4714285714285714",
      "accuracy": "0.795142555438226",
      "tn": "720",
      "fp": "157",
      "fn": "37",
      "tp": "33",
      "score_tipo": "margen_no_calibrado",
      "proyecto_test": "PC1",
      "proyectos_entrenamiento": "['CM1', 'JM1', 'KC1', 'KC2']",
      "escenario": "E8_ADASYN_RF",
      "algoritmo": "SVM",
      "cv_f1_macro_fuente": "0.44293816972916455",
      "vectores_test_presentes_en_train": "0",
      "modo": "con_deduplicacion_global",
      "validation_protocolo": "nested_leave_one_project_out",
      "elegido_procedimiento_general": "True"
    },
    {
      "n": "947",
      "positivos": "70",
      "prevalencia": "0.07391763463569166",
      "f1": "0.2914572864321608",
      "average_precision": "0.26333423765150704",
      "auc_roc": "0.7816826844762991",
      "precision": "0.2248062015503876",
      "recall": "0.4142857142857143",
      "accuracy": "0.8511087645195353",
      "tn": "777",
      "fp": "100",
      "fn": "41",
      "tp": "29",
      "score_tipo": "probabilidad_no_calibrada",
      "proyecto_test": "PC1",
      "proyectos_entrenamiento": "['CM1', 'JM1', 'KC1', 'KC2']",
      "escenario": "E2_ADASYN",
      "algoritmo": "Random Forest",
      "cv_f1_macro_fuente": "0.4060858656181452",
      "vectores_test_presentes_en_train": "0",
      "modo": "con_deduplicacion_global",
      "validation_protocolo": "nested_leave_one_project_out",
      "elegido_procedimiento_general": "False"
    },
    {
      "n": "947",
      "positivos": "70",
      "prevalencia": "0.07391763463569166",
      "f1": "0.25806451612903225",
      "average_precision": "0.22197761618601133",
      "auc_roc": "0.7513845903241572",
      "precision": "0.23529411764705882",
      "recall": "0.2857142857142857",
      "accuracy": "0.8785638859556494",
      "tn": "812",
      "fp": "65",
      "fn": "50",
      "tp": "20",
      "score_tipo": "probabilidad_no_calibrada",
      "proyecto_test": "PC1",
      "proyectos_entrenamiento": "['CM1', 'JM1', 'KC1', 'KC2']",
      "escenario": "E8_ADASYN_RF",
      "algoritmo": "XGBoost",
      "cv_f1_macro_fuente": "0.42901291461136876",
      "vectores_test_presentes_en_train": "0",
      "modo": "con_deduplicacion_global",
      "validation_protocolo": "nested_leave_one_project_out",
      "elegido_procedimiento_general": "False"
    }
  ],
  "seleccion_cv_previa": {
    "E6_ADASYN_MI": {
      "f1_macro": 0.41911964964178683,
      "ap_macro": 0.36036126620855324,
      "auc_macro": 0.7268613482845449,
      "folds": [
        {
          "proyecto_validacion": "CM1",
          "n": 439,
          "positivos": 46,
          "f1": 0.35294117647058826,
          "average_precision": 0.2903785462610261,
          "auc_roc": 0.7777409005420954,
          "precision": 0.24193548387096775,
          "recall": 0.6521739130434783
        },
        {
          "proyecto_validacion": "JM1",
          "n": 8787,
          "positivos": 1996,
          "f1": 0.41737360027146253,
          "average_precision": 0.34650755975165154,
          "auc_roc": 0.6509168019443392,
          "precision": 0.3155464340687532,
          "recall": 0.6162324649298597
        },
        {
          "proyecto_validacion": "KC1",
          "n": 1166,
          "positivos": 309,
          "f1": 0.42201834862385323,
          "average_precision": 0.385212010485827,
          "auc_roc": 0.642151253903698,
          "precision": 0.4,
          "recall": 0.44660194174757284
        },
        {
          "proyecto_validacion": "KC2",
          "n": 351,
          "positivos": 98,
          "f1": 0.6259541984732825,
          "average_precision": 0.5704273718557167,
          "auc_roc": 0.8039848350407357,
          "precision": 0.5,
          "recall": 0.8367346938775511
        },
        {
          "proyecto_validacion": "PC1",
          "n": 947,
          "positivos": 70,
          "f1": 0.2773109243697479,
          "average_precision": 0.20928084268854502,
          "auc_roc": 0.7595129499918554,
          "precision": 0.19642857142857142,
          "recall": 0.4714285714285714
        }
      ]
    },
    "E7_SMOTE_RF": {
      "f1_macro": 0.4146422654854806,
      "ap_macro": 0.3628130813088917,
      "auc_macro": 0.7259930223723162,
      "folds": [
        {
          "proyecto_validacion": "CM1",
          "n": 439,
          "positivos": 46,
          "f1": 0.3333333333333333,
          "average_precision": 0.25850221543119584,
          "auc_roc": 0.7665670981303242,
          "precision": 0.24489795918367346,
          "recall": 0.5217391304347826
        },
        {
          "proyecto_validacion": "JM1",
          "n": 8787,
          "positivos": 1996,
          "f1": 0.4320642682500873,
          "average_precision": 0.37095336914373844,
          "auc_roc": 0.6697610358399024,
          "precision": 0.33163538873994636,
          "recall": 0.6197394789579158
        },
        {
          "proyecto_validacion": "KC1",
          "n": 1166,
          "positivos": 309,
          "f1": 0.4065040650406504,
          "average_precision": 0.3993497109584745,
          "auc_roc": 0.6301427799994713,
          "precision": 0.4084967320261438,
          "recall": 0.4045307443365696
        },
        {
          "proyecto_validacion": "KC2",
          "n": 351,
          "positivos": 98,
          "f1": 0.6329113924050633,
          "average_precision": 0.5661291897624482,
          "auc_roc": 0.8001532628861823,
          "precision": 0.539568345323741,
          "recall": 0.7653061224489796
        },
        {
          "proyecto_validacion": "PC1",
          "n": 947,
          "positivos": 70,
          "f1": 0.2683982683982684,
          "average_precision": 0.2191309212486015,
          "auc_roc": 0.7633409350057012,
          "precision": 0.19254658385093168,
          "recall": 0.44285714285714284
        }
      ]
    }
  },
  "resumen_global_previo": {
    "sin_deduplicar": {
      "modo": "sin_deduplicar",
      "candidatos_por_busqueda": 27,
      "folds_externos": 5,
      "macro_f1_outer": 0.39900836430916486,
      "macro_ap_outer": 0.36068197160103477,
      "macro_auc_outer": 0.7554865270977444,
      "ganador_final": {
        "algoritmo": "SVM",
        "escenario": "E7_SMOTE_RF",
        "f1_cv_proyectos": 0.3972503492210537,
        "ap_cv_proyectos": 0.3284176195322347
      },
      "nota": "Nested outer estima procedimiento selección por proyecto; final es receta global reentrenada. Sin prueba inédita del joblib."
    },
    "con_deduplicacion_global": {
      "modo": "con_deduplicacion_global",
      "candidatos_por_busqueda": 27,
      "folds_externos": 5,
      "macro_f1_outer": 0.4122225477134025,
      "macro_ap_outer": 0.349042017450373,
      "macro_auc_outer": 0.7276729642842424,
      "ganador_final": {
        "algoritmo": "SVM",
        "escenario": "E6_ADASYN_MI",
        "f1_cv_proyectos": 0.41911964964178683,
        "ap_cv_proyectos": 0.36036126620855324
      },
      "nota": "Nested outer estima procedimiento selección por proyecto; final es receta global reentrenada. Sin prueba inédita del joblib."
    }
  }
}


## 5. Descargar los conjuntos de datos y validar su identidad

Los CSV CM1/JM1/KC1/KC2/PC1 se descargan desde URLs incluidas en el código, se comparan con SHA-256 exactos y se preparan con las mismas 21 métricas. Una coincidencia de hash demuestra identidad de archivos, **no certificación científica de origen**.

Si la descarga falla, Colab ofrecerá cargar el ZIP de respaldo que contenga los cinco CSV dentro de `01_datos_originales/`. Los archivos deben tener las huellas indicadas en el código; no se aceptan datos diferentes sin documentación y un nuevo plan de evaluación.

In [ ]:
import cpdp_core as base
import cpdp_evaluacion as evaluacion
from IPython.display import display

REF = json.loads(Path('referencia_experimental_incluida.json').read_text(encoding='utf-8'))
DATA_DIR.mkdir(parents=True,exist_ok=True)
fuentes=[]
try:
    for p in base.PROJECTS:
        filepath=DATA_DIR/f'{p}.csv'
        if not filepath.exists():
            url=f'{base.SOURCE_BASE}/{p.lower()}.csv'
            req=urllib.request.Request(url,headers={'User-Agent':'T2-CPDP-QA/1.0'})
            with urllib.request.urlopen(req,timeout=75) as response:
                content=response.read(25_000_001)
            if not 1000<len(content)<=25_000_000:
                raise RuntimeError(f'{p}: descarga incompleta o excesiva')
            filepath.write_bytes(content)
        actual=base.sha256(filepath)
        expected=REF['sha256_fuentes'][p]
        if actual!=expected:
            raise ValueError(f'{p}: SHA distinto al conjunto fijo de referencia (esperado {expected}, obtenido {actual}). NO mezclar fuentes.')
        fuentes.append({'proyecto':p,'url':f'{base.SOURCE_BASE}/{p.lower()}.csv','sha256':actual})
except Exception as exc:
    print('Descarga fallida o no compatible:',exc)
    print('Si tienes el ZIP de resultados con los cinco CSV, súbelo ahora. No importa si el fallo fue por restricciones de red.')
    from google.colab import files
    uploaded=files.upload()
    if len(uploaded)!=1: raise RuntimeError('Debes subir solo un archivo ZIP con 01_datos_originales/')
    zip_name=list(uploaded)[0]
    with zipfile.ZipFile(zip_name) as z:
        for p in base.PROJECTS:
            file_inside=f'01_datos_originales/{p}.csv'
            file_content=z.read(file_inside)
            filepath=DATA_DIR/f'{p}.csv'
            filepath.write_bytes(file_content)
            if base.sha256(filepath)!=REF['sha256_fuentes'][p]:
                raise ValueError(f'{p}: el CSV subido no coincide con la huella fijada')
    fuentes=[{'proyecto':p,'url':'ZIP de respaldo aportado por la tesista','sha256':base.sha256(DATA_DIR/f'{p}.csv')} for p in base.PROJECTS]

pd.DataFrame(fuentes).to_csv(OUT_DIR/'fuentes_y_sha256.csv',index=False)
original,audits = base.load_all(DATA_DIR)
print('Archivos verificados:',len(fuentes),'proyectos')
display(pd.DataFrame(audits)[['proyecto','filas','positivos','negativos','faltantes','sha256']])
print('Versiones:',{'sklearn':__import__('sklearn').__version__,
                    'imblearn':__import__('imblearn').__version__,
                    'xgboost':__import__('xgboost').__version__})


## 6. Decisión sobre duplicados y etiquetas contradictorias

**Cohorte principal:** conservar una sola fila para cada vector X idéntico de las 21 métricas, usando una selección determinista que **no consulta la etiqueta y**. Se registran conteos, huella y solapamiento entre proyectos. La elección de la fila representativa no resuelve por sí sola la inconsistencia de etiquetas; es una decisión metodológica que se debe defender.

**Sensibilidad secundaria:** descartar todos los grupos en que un mismo vector X está asociado tanto a 0 como a 1, y después deduplicar. Este análisis **sí utiliza las etiquetas del corpus completo** como limpieza retrospectiva y **no simula un filtro aplicable a un proyecto nuevo sin etiquetas**.

Se guardan ambos resultados para evaluar la sensibilidad de las conclusiones. Debe completarse en el documento de tesis la referencia científica de las fuentes y la justificación de la regla escogida.

In [ ]:
evaluacion.quick_selfcheck()
cohortes, auditoria = evaluacion.build_cohorts(original,OUT_DIR)
assert auditoria['huella_principal']==REF['hash_proceso_dedup'], 'La cohorte principal no coincide con la huella registrada; revisar datos y política.'
print('Registros originales:',auditoria['original_total'])
print('Principal (una fila por vector X):',auditoria['principal_total'])
print('Sin grupos contradictorios:',auditoria['sensibilidad_total'])
print('Grupos contradictorios:',auditoria['conflictos']['grupos_contradictorios'])
print('Filas involucradas:',auditoria['conflictos']['filas_en_grupos_contradictorios'])


## 7. Evaluación de las recetas E6 y E7

Se reserva un proyecto completo como prueba, se entrena cada receta fija con los otros cuatro y se registran F1, precision, recall, AP y AUC. Se repite en la cohorte principal y en la cohorte de sensibilidad.

La prueba de Wilcoxon bilateral incluida aquí es **solo exploratoria**: con cinco pares no se puede obtener p < 0,05 cuando los cinco signos son concordantes (p mínimo bilateral exacto = 0,0625). Que el resultado no sea significativo **no prueba equivalencia**. Los análisis nuevos por esfuerzo añadirán diferencias descriptivas por proyecto e intervalos bootstrap condicionales.

In [ ]:
tablas_pareadas=[]
qa_tables=[]
if RUN_FIJO_E6_E7:
    for cohorte in ['principal_deduplicada','eliminar_grupos_conflictivos']:
        rr, qa = evaluacion.evaluate_fixed_configs(cohortes[cohorte],cohorte,OUT_DIR/f'comparacion_fija_{cohorte}')
        paired=evaluacion.paired_tests(rr,cohorte,OUT_DIR/'pruebas_estadisticas')
        tablas_pareadas.append(paired)
        qa_tables.append(qa)
        print('\nRESUMEN',cohorte)
        display(paired[['metrica','media_E6','media_E7','media_diferencia_E6_menos_E7',
                        'p_bilateral_exacto_exploratorio','proyectos_E6_mayor','proyectos_E7_mayor']])
        display(rr.pivot(index='proyecto',columns='escenario',values='f1'))
        display(rr.groupby('escenario')[['segundos_entrenamiento','segundos_inferencia_evaluacion']].agg(['mean','std']))
    pd.concat(tablas_pareadas,ignore_index=True).to_csv(OUT_DIR/'prueba_E6_vs_E7_resumen.csv',index=False)
else:
    print('Comparación E6/E7 omitida por configuración, no se han producido resultados nuevos.')


## 8. Reentrenar las recetas seleccionadas por partición

Se usan las configuraciones incluidas en la tabla de referencia para volver a entrenar el modelo de cada proyecto de prueba. Se mantiene una separación clara entre **reejecución de recetas ya seleccionadas** y **nueva selección interna de 27 candidatos**, que no está activada por defecto.

Esta sección complementa la descripción científica del estudio. **La estrategia de riesgo/LOC que pide el asesor se implementa después con E6, calibración por proyectos y pruebas específicas**, sin reutilizar resultados numéricos de esta sección.
**Nota de ejecución:** si las dos comparaciones opcionales están desactivadas, se omiten sin error. La evaluación solicitada por el profesor se ejecuta después, en la sección 10.


In [ ]:
# Esta sección es OPCIONAL: reproduce configuraciones seleccionadas en un análisis anterior.
# La nueva estrategia del profesor se ejecuta en la sección 10 sin depender de estos resultados.
if REPRODUCIR_SELECCIONES_PREVIAS:
    for cohorte, ref_key in [('sin_deduplicar', 'sin_deduplicar'),
                             ('principal_deduplicada', 'con_deduplicacion_global')]:
        rr, qa = evaluacion.evaluate_reference_nested(
            cohortes[cohorte], cohorte,
            REF['seleccion_previa_por_fold'][ref_key],
            OUT_DIR / f'procedimiento_anidado_{cohorte}'
        )
        qa_tables.append(qa)
        print('Selecciones de referencia reentrenadas en:', cohorte)
        display(rr[['proyecto', 'algoritmo', 'escenario', 'f1',
                    'average_precision', 'auc_roc']])
else:
    print('Comparación de selecciones de referencia omitida (opcional).')

# No hay error cuando ambas evaluaciones opcionales están desactivadas.
# En ese caso se continúa hacia el entrenamiento y la calibración E6/LOC.
if qa_tables:
    summary = evaluacion.summarize_rankings(qa_tables, OUT_DIR)
    resumen_interes = summary.loc[
        (summary['presupuesto_fraccion'].isin([.1, .2])) &
        (summary['procedimiento'] == 'seleccion_previa_reentrenada'),
        ['cohorte', 'metodo', 'presupuesto_fraccion',
         'recall_por_modulos', 'recall_por_LOC_modulos_completos',
         'popt_normalizado_0_20', 'popt_normalizado_0_100']
    ]
    if not resumen_interes.empty:
        display(resumen_interes)
    else:
        print('Sin comparaciones de selecciones previas; se continúa con estrategia E6/LOC.')
else:
    print('Sin evaluaciones previas nuevas: se continúa con la estrategia del profesor ' 
          '(probabilidad calibrada E6 / LOC), que se ejecuta en la sección 10.')

# Estas tablas son REFERENCIAS HISTÓRICAS incrustadas, no resultados nuevos.
pd.DataFrame(REF['comparacion_algoritmos_externa_previa']).to_csv(
    OUT_DIR / 'comparacion_algoritmos_referencia_previa.csv', index=False)
pd.DataFrame(REF['comparacion_global_previa']).to_csv(
    OUT_DIR / 'comparacion_27_configuraciones_referencia_previa.csv', index=False)


## 9. Ranking existente y gráficos comparativos

Aquí se muestran las reglas de referencia: **margen ML**, **ManualUp** (LOC ascendente), **LOC descendente** y **aleatorio esperado**, bajo fracciones de 10 %, 20 % y 30 % del esfuerzo en LOC. También se conserva el presupuesto por número de módulos y la métrica Popt.

Los resultados deben mostrarse incluso cuando el ML pierda frente a ManualUp. Una diferencia favorable bajo un presupuesto por módulos **no puede usarse como sustituto** del resultado bajo presupuesto por LOC.

In [ ]:
import matplotlib.pyplot as plt
if qa_tables:
    resumen = evaluacion.summarize_rankings(qa_tables, OUT_DIR)
    qa_all = pd.concat(qa_tables, ignore_index=True)
    qa_all.to_csv(OUT_DIR/'QA_por_proyecto_y_procedimiento.csv', index=False)
    resumen_interes=(resumen.loc[(resumen['presupuesto_fraccion'].isin([.1,.2])) &
                (resumen['procedimiento']=='seleccion_previa_reentrenada')]
        [['cohorte','metodo','presupuesto_fraccion','recall_por_modulos',
          'recall_por_LOC_modulos_completos','popt_normalizado_0_20','popt_normalizado_0_100']])
    display(resumen_interes)
    for cohorte in ['sin_deduplicar','principal_deduplicada']:
        focus=resumen[(resumen.cohorte==cohorte)&(resumen.procedimiento=='seleccion_previa_reentrenada')]
        if focus.empty: continue
        fig,ax=plt.subplots(figsize=(8,4))
        for name in ['ML','ManualUp','LOC_desc','Aleatorio_esperado']:
            f=focus[focus.metodo==name].sort_values('presupuesto_fraccion')
            ax.plot(f.presupuesto_fraccion*100,f.recall_por_LOC_modulos_completos*100,marker='o',label=name)
        ax.set_xlabel('Presupuesto de revisión (% LOC)');ax.set_ylabel('Recall de módulos defectuosos (%)')
        ax.set_title('Reglas de referencia: '+cohorte);ax.grid(alpha=.25);ax.legend()
        fig.tight_layout();fig.savefig(OUT_DIR/f'grafico_recall_LOC_{cohorte}.png',dpi=180)
        plt.show()
else:
    print('La repetición de comparaciones previas se omitió; la nueva estrategia de riesgo/LOC se ejecuta en la sección 10.')

# Tablas de antecedente, no producidas por un entrenamiento nuevo en esta celda.
pd.DataFrame(REF['comparacion_algoritmos_externa_previa']).to_csv(
    OUT_DIR/'comparacion_algoritmos_referencia_previa.csv',index=False)
pd.DataFrame(REF['comparacion_global_previa']).to_csv(
    OUT_DIR/'comparacion_27_configuraciones_referencia_previa.csv',index=False)
print('Popt_0_20: curva truncada al 20%; Popt_0_100: curva completa normalizada.')

## 10. Nueva estrategia solicitada por el asesor: **probabilidad calibrada / LOC**

### Problema que se corrige

La receta **E6** usa un SVM que produce un **margen** (`decision_function`). Ordenar directamente por ese margen puede priorizar módulos grandes que consumen gran parte del presupuesto. El margen **no es una probabilidad** y no debe dividirse por LOC como si lo fuera.

### Fórmula del nuevo método

Para cada módulo i se calcula:

\\[\\operatorname{densidad\_riesgo}_i=\\frac{\\widehat{P}(y_i=1\\mid X_i)}{\\max(\\operatorname{LOC}_i,1)}\\]

Se inspeccionan primero los módulos con mayor densidad de riesgo. El denominador `max(LOC,1)` evita divisiones por cero y se utiliza igualmente al estimar el costo de todas las líneas base.

### Cómo se obtiene la probabilidad **sin fuga de datos**

Para cada proyecto de prueba, por ejemplo CM1:

1. **CM1 no participa en ningún entrenamiento ni calibración.** Los proyectos fuente son JM1, KC1, KC2 y PC1.
2. Se generan márgenes *out-of-fold* sobre los cuatro proyectos fuente: para obtener márgenes de JM1, E6 se entrena solo con KC1+KC2+PC1; se repite dejando fuera cada una de las cuatro fuentes.
3. Se ajusta una **regresión logística sigmoide** con los márgenes obtenidos en las fuentes y sus etiquetas. No se utilizan resultados del proyecto de prueba.
4. Se entrena E6 con los cuatro proyectos fuente completos, se calcula su margen sobre CM1 y se lo transforma con la sigmoide ajustada. Se obtienen probabilidades en [0,1].
5. Se calcula `probabilidad / max(LOC,1)` y se compara con **ManualUp** y con el ranking ML ordinario usando el **mismo conjunto de prueba y presupuesto del 20 % de LOC**.
6. Se repite lo anterior con cada proyecto como prueba. Se registran las cinco diferencias y un **bootstrap por módulos dentro de cada proyecto**, con los scores fijos, para mostrar incertidumbre condicional.

### Comparación y cautela estadística

- **Comparación principal de este análisis adicional:** `ML_prob_calibrada_por_LOC` frente a `ManualUp`, usando `Recall@20 % LOC` y solo módulos totalmente inspeccionados.
- **Controles:** `ML_margen_SVM`, `ML_prob_calibrada` sin división por LOC, `LOC_desc` y orden aleatorio esperado. También se reportan `Recall@10/30 % LOC`, `Recall@20 % módulos` y Popt.
- Un criterio descriptivo exigente para comentar una posible mejora es obtener **Δ positivo en los 5 proyectos**, idealmente con el límite inferior del intervalo bootstrap de cada uno por encima de cero; **no** es una prueba confirmatoria.
- Como ya se han observado resultados de estos proyectos, se trata de un **análisis adicional exploratorio** motivado por una observación del asesor. No debe presentarse como una hipótesis preregistrada ni modifica retrospectivamente el objetivo o el plan inicial.
- Los intervalos bootstrap condicionan en predicciones fijas y no reflejan incertidumbre por un sexto proyecto o por volver a entrenar los modelos. **Un resultado negativo es válido y debe mostrarse.**

A continuación se crea íntegramente el script de calibración y comparación. No necesita archivos Python externos.

In [ ]:
%%writefile cpdp_estrategia_esfuerzo.py
"""Priorización por probabilidad CALIBRADA / LOC, sin usar etiquetas del proyecto test.

Calibración por proyecto fuente: en cada uno de los cuatro proyectos fuente se
predice con un SVM E6 ajustado solo en los otros tres. Con los márgenes OOF
(out-of-fold) y sus etiquetas se ajusta una regresión logística sigmoide
(Platt como mapeo). Luego se reentrena E6 en los cuatro proyectos fuente.
El proyecto objetivo no participa ni en los ajustes ni en la calibración.

No cambia el modelo del prototipo; este procedimiento se usa para evaluar
priorización. El resultado es exploratorio: se conocían datos anteriores.
"""
from __future__ import annotations
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, average_precision_score, roc_auc_score
import cpdp_core as base
import cpdp_evaluacion as evaluacion

SEED = base.SEED
BUDGET = 0.20
CALIBRATION_METHOD = 'sigmoide_logistica_en_margenes_OOF_por_proyecto'
METHODS = ['ML_margen_SVM','ML_prob_calibrada','ML_prob_calibrada_por_LOC',
           'ManualUp','LOC_desc','Aleatorio_esperado']


def registrar_plan_analisis(out_dir):
    """Registrar antes de esta ejecución; NO afirmar preregistro histórico."""
    out_dir = Path(out_dir);out_dir.mkdir(parents=True,exist_ok=True)
    plan = {
        'caracter': 'Analisis adicional exploratorio formulado tras conocer resultados de cinco proyectos; no confirmatorio ni preregistrado',
        'modelo_fijo': 'E6_ADASYN_MI / SVM',
        'proyectos': list(base.PROJECTS),
        'cohorte_primaria': 'principal_deduplicada: una X global representativa sin consultar y',
        'estrategia_nueva': 'mayor probabilidad calibrada de defecto dividida entre max(LOC, 1)',
        'calibracion': CALIBRATION_METHOD,
        'comparacion_primaria_nueva': 'ML_prob_calibrada_por_LOC frente a ManualUp, Recall@20 % LOC, modulos completos',
        'indicadores_secundarios': ['Recall@10 % LOC','Recall@30 % LOC','Recall@20 % modulos','Popt normalizado 0-20','Popt normalizado 0-100','Brier de calibracion'],
        'unidad_primaria': 'proyecto (cinco proyectos independientes a nivel conceptual, con amenazas de calidad NASA)',
        'resumen_descriptivo': ['diferencia por proyecto','mediana','numero de proyectos con diferencia positiva','intervalo bootstrap por proyecto'],
        'criterio_descriptivo_estricto': 'Cinco de cinco deltas positivos; para apoyo adicional, limite inferior bootstrap de cada proyecto > 0',
        'limitaciones': ['El intervalo bootstrap re-muestrea modulos y condiciona en los modelos entrenados; no captura incertidumbre por nuevos proyectos.',
                        'Cinco proyectos no permiten concluir superioridad estadistica con Wilcoxon bilateral exacto al 5%.',
                        'No cambiar retrospectivamente el criterio primario del plan original; documentar este analisis adicional como desviacion motivada por observacion del asesor.',
                        'El costo LOC es una aproximacion del esfuerzo de QA, no un tiempo real medido.']}
    evaluacion.json_save(out_dir/'plan_analisis_estrategia_esfuerzo.json',plan)
    return plan


def calibrar_en_fuentes(cohortes, proyecto_objetivo):
    """Devuelve los scores de TEST; calibración solo con cuatro proyectos fuente."""
    fuentes = [p for p in base.PROJECTS if p != proyecto_objetivo]
    assert len(fuentes)==4 and proyecto_objetivo not in fuentes
    oof_scores=[];oof_y=[];tablas_cal=[]
    tic=time.perf_counter()
    for proyecto_cal in fuentes:
        entrena=[p for p in fuentes if p!=proyecto_cal]
        assert proyecto_objetivo not in entrena and proyecto_cal not in entrena
        Xent,yent=base.pack(cohortes,entrena)
        Xcal,ycal=cohortes[proyecto_cal]
        svm=base.make_pipeline('E6_ADASYN_MI','SVM')
        svm.fit(Xent,yent)
        margenes=np.asarray(svm.decision_function(Xcal),dtype=float)
        if not np.isfinite(margenes).all():
            raise ValueError('Margenes no finitos en calibracion')
        oof_scores.append(margenes)
        oof_y.append(np.asarray(ycal,dtype=int))
        tablas_cal.append({'proyecto_objetivo':proyecto_objetivo,
            'proyecto_calibracion':proyecto_cal,'proyectos_entrenamiento':','.join(entrena),
            'n_calibracion':len(ycal),'positivos_calibracion':int(ycal.sum())})
    xx=np.concatenate(oof_scores).reshape(-1,1)
    yy=np.concatenate(oof_y)
    if len(np.unique(yy))!=2:
        raise ValueError('La calibracion requiere ambas clases')
    calibrador=LogisticRegression(solver='lbfgs',max_iter=2000,C=1.0,
                                 random_state=SEED)
    calibrador.fit(xx,yy)
    Xtrain,ytrain=base.pack(cohortes,fuentes)
    Xtest,ytest=cohortes[proyecto_objetivo]
    modelo_final=base.make_pipeline('E6_ADASYN_MI','SVM')
    modelo_final.fit(Xtrain,ytrain)
    scores=np.asarray(modelo_final.decision_function(Xtest),dtype=float)
    prob=np.asarray(calibrador.predict_proba(scores.reshape(-1,1))[:,1],dtype=float)
    if not np.isfinite(prob).all() or np.any((prob<0)|(prob>1)):
        raise ValueError('Probabilidades calibradas fuera del rango [0,1]')
    metadatos={'proyecto':proyecto_objetivo,'fuentes_calibracion':','.join(fuentes),
       'n_OOF_calibrador':int(len(yy)),
       'n_positivos_OOF':int(yy.sum()),
       'coeficiente_sigmoide':float(calibrador.coef_[0,0]),
       'intercepto_sigmoide':float(calibrador.intercept_[0]),
       'segundos_ajuste_y_calibracion':float(time.perf_counter()-tic),
       'brier_test':float(brier_score_loss(np.asarray(ytest,dtype=int),prob)),
       'ap_test_prob_calibrada':float(average_precision_score(ytest,prob)),
       'auc_test_prob_calibrada':float(roc_auc_score(ytest,prob)),
       'advertencia':'Calibrador entrenado con OOF de fuentes; el proyecto test nunca interviene. Brier test reporta calibracion, no valida un nuevo proyecto externo.'}
    return scores,prob,metadatos,tablas_cal


def ranking_con_presupuesto(y,loc,puntuacion,presupuesto=BUDGET,ascendente=False):
    """Recall de módulos COMPLETOS; costo mínimo por módulo = 1 LOC."""
    y=np.asarray(y,dtype=int);loc=np.maximum(np.asarray(loc,dtype=float),1.)
    score=np.asarray(puntuacion,dtype=float)
    if not (len(y)==len(loc)==len(score)) or len(y)==0:
        raise ValueError('longitud incompatible')
    if not np.isfinite(loc).all() or not np.isfinite(score).all():
        raise ValueError('puntuacion/LOC no finitos')
    if y.sum()==0:
        return np.nan
    ids=np.arange(len(y))
    orden=np.lexsort((ids,score if ascendente else -score))
    presupuesto_loc=presupuesto*float(loc.sum())
    usados=np.cumsum(loc[orden])
    completos=int(np.searchsorted(usados,presupuesto_loc,side='right'))
    return float(y[orden[:completos]].sum()/y.sum())


def bootstrap_delta_proyecto(y,loc,p,iteraciones=1000,seed=SEED):
    """Intervalo condicional: remuestrea módulos, no reentrena el modelo.

    Si una réplica no contiene defectos, se descarta; se documenta cuántas.
    """
    y=np.asarray(y,dtype=int);loc=np.maximum(np.asarray(loc,dtype=float),1.)
    p=np.asarray(p,dtype=float)
    rng=np.random.default_rng(seed)
    deltas=[];omitidos=0
    for _ in range(iteraciones):
        idx=rng.integers(0,len(y),size=len(y))
        ys=y[idx];ls=loc[idx];ps=p[idx]
        if ys.sum()==0:
            omitidos+=1;continue
        a=ranking_con_presupuesto(ys,ls,ps/ls)
        b=ranking_con_presupuesto(ys,ls,ls,ascendente=True)
        deltas.append(a-b)
    if len(deltas)<int(.95*iteraciones):
        raise ValueError('Demasiadas replicas sin positivos para bootstrap estable')
    return {'bootstrap_repeticiones':int(iteraciones),
            'bootstrap_replicas_validas':len(deltas),
            'bootstrap_replicas_sin_positivos':omitidos,
            'IC95_delta_inferior':float(np.quantile(deltas,.025)),
            'IC95_delta_superior':float(np.quantile(deltas,.975)),
            'nota_IC':'Remuestreo de modulos condicionado en scores fijos; no representa incertidumbre entre proyectos ni reentrenamiento.'}


def ejecutar(cohortes,cohorte,root,bootstrap=1000):
    """Comparación primaria de probabilidad/LOC con ManualUp y controles."""
    root=Path(root);root.mkdir(parents=True,exist_ok=True)
    scores_por_proyecto=[];ranking=[];calibraciones=[];cal_folds=[];intervalos=[]
    for proyecto in base.PROJECTS:
        print('Calibración y evaluación E6 por esfuerzo:',cohorte,proyecto,flush=True)
        margen,prob,meta,folds=calibrar_en_fuentes(cohortes,proyecto)
        Xtest,ytest=cohortes[proyecto]
        y=np.asarray(ytest,dtype=int)
        loc=np.maximum(np.asarray(Xtest['loc'],dtype=float),1.)
        densidad=prob/loc
        registro=pd.DataFrame({'proyecto':proyecto,'indice_local':np.arange(len(y)),
            'etiqueta_defecto':y,'loc_costo':loc,'margen_SVM_E6':margen,
            'probabilidad_defecto_calibrada':prob,'riesgo_por_LOC':densidad})
        scores_por_proyecto.append(registro)
        calibraciones.append(meta);cal_folds.extend(folds)
        for nombre,scores in [('ML_margen_SVM',margen),
                              ('ML_prob_calibrada',prob),
                              ('ML_prob_calibrada_por_LOC',densidad)]:
            filas=evaluacion.ranking_evaluation(y,scores,loc,project=proyecto,
                    cohort=cohorte,procedimiento='E6_calibrado_por_proyectos',random_repeats=50)
            for row in filas:
                if row['metodo']=='ML':
                    row['metodo']=nombre;ranking.append(row)
                elif nombre=='ML_margen_SVM':
                    ranking.append(row)  # ManualUp, LOC-desc, aleatorio: solo una vez.
        ci=bootstrap_delta_proyecto(y,loc,prob,iteraciones=bootstrap,
                seed=SEED+sum(ord(x) for x in proyecto))
        intervalos.append({'proyecto':proyecto,'cohorte':cohorte,**ci})
    pred=pd.concat(scores_por_proyecto,ignore_index=True)
    tablas=pd.DataFrame(ranking)
    cal_df=pd.DataFrame(calibraciones)
    ci_df=pd.DataFrame(intervalos)
    pred.to_csv(root/'predicciones_por_proyecto.csv',index=False)
    tablas.to_csv(root/'ranking_por_proyecto_y_presupuesto.csv',index=False)
    cal_df.to_csv(root/'calibracion_por_proyecto.csv',index=False)
    pd.DataFrame(cal_folds).to_csv(root/'folds_calibracion_solo_fuentes.csv',index=False)
    ci_df.to_csv(root/'IC95_por_proyecto.csv',index=False)
    summary=(tablas.groupby(['metodo','presupuesto_fraccion'],as_index=False)
        [['recall_por_modulos','recall_por_LOC_modulos_completos',
          'popt_normalizado_0_20','popt_normalizado_0_100']].mean())
    summary.to_csv(root/'resumen_macro_por_metodo_y_presupuesto.csv',index=False)
    # Contraste por proyecto de la métrica operacional 20 % LOC.
    focus=tablas[np.isclose(tablas.presupuesto_fraccion,BUDGET)]
    pivot=focus.pivot(index='proyecto',columns='metodo',values='recall_por_LOC_modulos_completos')
    per=pd.DataFrame({'proyecto':pivot.index,
          'ML_margen':pivot['ML_margen_SVM'].to_numpy(),
          'ML_probabilidad':pivot['ML_prob_calibrada'].to_numpy(),
          'ML_probabilidad_por_LOC':pivot['ML_prob_calibrada_por_LOC'].to_numpy(),
          'ManualUp':pivot['ManualUp'].to_numpy(),
          'LOC_desc':pivot['LOC_desc'].to_numpy()})
    per['delta_ML_calibrado_por_LOC_menos_ManualUp']=per['ML_probabilidad_por_LOC']-per['ManualUp']
    per=per.merge(ci_df[['proyecto','IC95_delta_inferior','IC95_delta_superior']],on='proyecto',how='left',validate='one_to_one')
    per.to_csv(root/'diferencias_20pct_LOC_por_proyecto.csv',index=False)
    d=per['delta_ML_calibrado_por_LOC_menos_ManualUp'].to_numpy()
    criterio=bool(np.all(d>0) and np.all(per['IC95_delta_inferior']>0))
    informe={'comparacion':'ML_prob_calibrada_por_LOC menos ManualUp en Recall@20% LOC',
          'cohorte':cohorte,'n_proyectos':5,
          'promedio_ml_calibrado_por_loc':float(per.ML_probabilidad_por_LOC.mean()),
          'promedio_manualup':float(per.ManualUp.mean()),
          'mediana_delta':float(np.median(d)),
          'media_delta':float(np.mean(d)),
          'proyectos_delta_positivo':int(np.sum(d>0)),
          'proyectos_delta_negativo':int(np.sum(d<0)),
          'proyectos_empate':int(np.sum(d==0)),
          'cumple_criterio_descriptivo_estricto':criterio,
          'advertencia':'Resultado exploratorio. No constituye prueba de superioridad externa ni se cambia el criterio segun el resultado; delta por proyecto e IC condicionales.'}
    evaluacion.json_save(root/'decision_descriptiva.json',informe)
    # Imagen comparativa y gráfica de diferencias con CI por proyecto.
    fig,ax=plt.subplots(figsize=(10,5))
    for nombre in ['ML_margen_SVM','ML_prob_calibrada','ML_prob_calibrada_por_LOC','ManualUp']:
        subset=summary[summary.metodo==nombre].sort_values('presupuesto_fraccion')
        ax.plot(subset.presupuesto_fraccion.to_numpy()*100,
                subset.recall_por_LOC_modulos_completos.to_numpy()*100,
                marker='o',label=nombre)
    ax.set(title='Priorización E6 frente a ManualUp (presupuesto en LOC)',
           xlabel='Presupuesto de LOC inspeccionadas (%)',
           ylabel='Recall de módulos defectuosos (%)')
    ax.grid(alpha=.3);ax.legend(fontsize=8);fig.tight_layout()
    fig.savefig(root/'curvas_recall_LOC.png',dpi=180)
    plt.close(fig)
    fig,ax=plt.subplots(figsize=(9,5))
    xs=np.arange(len(per));vs=per.delta_ML_calibrado_por_LOC_menos_ManualUp.to_numpy()*100
    lower=(per.delta_ML_calibrado_por_LOC_menos_ManualUp-per.IC95_delta_inferior).to_numpy()*100
    upper=(per.IC95_delta_superior-per.delta_ML_calibrado_por_LOC_menos_ManualUp).to_numpy()*100
    ax.errorbar(xs,vs,yerr=np.vstack([np.maximum(lower,0),np.maximum(upper,0)]),
                fmt='o',capsize=4)
    ax.axhline(0,color='gray',linestyle='--')
    ax.set_xticks(xs,labels=per.proyecto.to_list())
    ax.set(xlabel='Proyecto reservado',ylabel='Δ Recall@20 % LOC (puntos porcentuales)',
           title='ML calibrado/LOC menos ManualUp; IC 95 % bootstrap condicional')
    ax.grid(axis='y',alpha=.3);fig.tight_layout()
    fig.savefig(root/'delta_por_proyecto_IC95.png',dpi=180)
    plt.close(fig)
    return per,summary,cal_df,informe


def selfcheck():
    y=np.array([0,1,0,1,0,1,0,0,1],dtype=int)
    loc=np.array([100,2,80,3,20,4,50,10,5],dtype=float)
    prob=np.array([.2,.7,.1,.9,.2,.5,.8,.1,.75])
    base_recall=ranking_con_presupuesto(y,loc,loc,ascendente=True)
    new_recall=ranking_con_presupuesto(y,loc,prob/loc)
    assert 0<=base_recall<=1 and 0<=new_recall<=1
    ci=bootstrap_delta_proyecto(y,loc,prob,iteraciones=100,seed=SEED)
    assert ci['bootstrap_replicas_validas']>90
    print('Pruebas unitarias de presupuesto, ranking y bootstrap: OK')


### Ejecutar el ranking calibrado y guardar evidencias

Primero se guarda el criterio de comparación con una marca explícita de análisis exploratorio. Después se evalúan los cinco proyectos de la **cohorte principal**. Para limitar el tiempo de ejecución, la evaluación del nuevo ranking sobre la cohorte de sensibilidad está desactivada de inicio; los resultados de sensibilidad de **clasificación E6/E7** se obtienen igualmente en la sección anterior.

Para evaluar también la nueva estrategia de riesgo/LOC sobre la cohorte sin contradicciones, cambia `EJECUTAR_SENSIBILIDAD_RANKING = True` antes de ejecutar. Cada proyecto genera probabilidades nuevas mediante calibración realizada exclusivamente con sus proyectos fuente. Se guardan CSV, JSON y gráficos con evidencia reproducible.

In [ ]:
import cpdp_estrategia_esfuerzo as estrategia
EJECUTAR_SENSIBILIDAD_RANKING = False  # Opcional: True para probar el nuevo ranking en ambas cohortes
REPETICIONES_BOOTSTRAP = 1000

PLAN_QA = estrategia.registrar_plan_analisis(OUT_DIR/'estrategia_esfuerzo')
estrategia.selfcheck()
comprobar = ['principal_deduplicada']
if EJECUTAR_SENSIBILIDAD_RANKING:
    comprobar.append('eliminar_grupos_conflictivos')

resultados_esfuerzo = {}
for nombre_cohorte in comprobar:
    delta, resumen_qa, calibracion, informe = estrategia.ejecutar(
        cohortes[nombre_cohorte],nombre_cohorte,
        OUT_DIR/'estrategia_esfuerzo'/nombre_cohorte,
        bootstrap=REPETICIONES_BOOTSTRAP)
    resultados_esfuerzo[nombre_cohorte]=informe
    print('\nCOMPARACIÓN PRINCIPAL (Recall@20 % LOC) ·', nombre_cohorte)
    display(delta.round(4))
    print('RESUMEN DESCRIPTIVO:', informe)
    display(calibracion[['proyecto','n_OOF_calibrador','coeficiente_sigmoide','brier_test']].round(4))

print('Archivos generados dentro de:', OUT_DIR/'estrategia_esfuerzo')

## 11. Interpretación del clasificador E6 y comparación con E7

El prototipo utiliza **E6 (SVM + ADASYN + selección por información mutua)**. E7 se estudia como alternativa para examinar diferencias en F1/AP/AUC y sensibilidad a las etiquetas contradictorias. **No se cambia automáticamente el artefacto desplegado ni sus manifiestos**.

No debe elegirse una configuración nueva usando las etiquetas del proyecto de prueba y después presentarla como una prueba externa independiente. La comparación que sigue evalúa una nueva **regla de priorización**; no implica cambiar el clasificador de producción sin revisión técnica.

## 12. Estado del artefacto y limitaciones

- El SVM E6 del prototipo produce un **margen de decisión**, no una probabilidad calibrada.
- La nueva estrategia de esta investigación añade calibración de dicho margen en un proceso de evaluación **separado**, sin sobrescribir el modelo ni su API de inferencia.
- No se dispone de un sexto proyecto externo y completamente inédito; los resultados son comparaciones sobre los cinco proyectos históricos.
- La conclusión debe separar clasificación de defectos (F1, AUC, AP) y priorización por costo de inspección (Recall@LOC, Popt).

In [ ]:
if EXPORTAR_DOS_CANDIDATOS:
    exported=evaluacion.export_model_candidates(cohortes['principal_deduplicada'],OUT_DIR/'modelos_candidatos',
                                         REF['sha256_fuentes'])
    display(exported)
else:
    print('Se omitió la exportación de candidatos por configuración.')

evaluacion.json_save(OUT_DIR/'LEEME_PROTOCOLO.json',{
  'protocolo':evaluacion.EXPERIMENTO_ID,
  'referencia_incorporada':'Selecciones experimentales anteriores incluidas en el notebook; verificación mediante SHA-256 y huella deduplicada',
  'metrica_clasificacion_primaria':'F1 positivo por proyecto; AP/AUC secundarios',
  'metrica_QA_1':'Recall@10/20/30% módulos',
  'metrica_QA_2':'Recall con presupuestos 10/20/30% LOC, módulos completos',
  'lineas_base':['LOC_desc','ManualUp','Aleatorio_esperado'],
  'popt':'Popt_[0,b] = (A_modelo-A_worst)/(A_optimo-A_worst), b=.2 o 1.0; curvas con interpolación; el recall operativo requiere módulos completos',
  'contradicciones':'Benchmark histórico: principal sin usar etiqueta para elegir poseedor; sensibilidad elimina grupos conflictivos usando y de benchmark completo; no prueba ciega',
  'p_value':'Wilcoxon exacto exploratorio con n=5. Ausencia de diferencia significativa no implica equivalencia.',
  'limites':['Reentrenamiento de selecciones previas por proyecto no evalúa el joblib final',
            'E6/E7 evaluados en cinco proyectos previamente usados para selección',
            'Lenguajes y procedencia documental no verificados; transferibilidad sin probar',
            'No se garantiza mejora frente a ManualUp',
            'No hay extracción automática de 21 métricas en Streamlit'],
  'modelo_streamlit_actual':'E6_ADASYN_MI; integración verificada por el asesor el 02/11/2026; resultados de nueva ejecución deben verificarse',
  'candidatos_exportados':'No utilizar automáticamente como versión de producción',
  'referencias_metodologicas':[
     'Mende y Koschke (2010), Effort-Aware Defect Prediction Models',
     'Shepperd et al. (2013), Data Quality: Some Comments on the NASA Software Defect Data Sets',
     'Yu et al. (2019), Effort-aware and just-in-time defect prediction with neural network' ]
})
print('Evidencias guardadas en:',OUT_DIR)


## 13. Búsqueda de 27 configuraciones (opcional)

El procedimiento costoso de selección de 27 combinaciones de clasificadores, balanceo y selección de variables está incluido en el cuaderno, pero **no se ejecuta por defecto**. Para el avance actual, lo importante es evaluar la estrategia de probabilidad calibrada / LOC con E6, no repetir sin necesidad todos los entrenamientos.

In [ ]:
if REPETIR_BUSQUEDA_COMPLETA_27:
    for cohort_name in ['principal_deduplicada','eliminar_grupos_conflictivos']:
        print('Búsqueda anidada 27 candidatos:',cohort_name)
        base.run_nested(cohortes[cohort_name],audits,
            OUT_DIR/f'busqueda_completa_27_{cohort_name}',modo=cohort_name)
else:
    print('Búsqueda de 27 candidatos OMITIDA. Se reutilizaron únicamente las selecciones previas incorporadas y verificadas.')


## 14. Generar ZIP final para revisión y GitHub

La salida contendrá tablas nuevas de métricas, auditoría de datos, gráficos, código Python íntegro y un archivo de criterios que identifica el análisis adicional como exploratorio. **Las referencias históricas que se incluyen están marcadas como antecedentes**; no se confunden con cifras calculadas por el presente experimento.

El ZIP no actualiza GitHub ni el modelo de Streamlit automáticamente. Descarga también este `.ipynb` ejecutado para conservar las celdas, tablas y salidas.

In [ ]:
from zipfile import ZipFile,ZIP_DEFLATED
out_zip=Path('/content/T2_CPDP_Resultados_y_Evidencias.zip')
with ZipFile(out_zip,'w',ZIP_DEFLATED,compresslevel=6) as z:
    for p in OUT_DIR.rglob('*'):
        if p.is_file():z.write(p,arcname=p.relative_to(DIRECTORIO).as_posix())
    for p in ['cpdp_core.py','cpdp_evaluacion.py','cpdp_estrategia_esfuerzo.py','referencia_experimental_incluida.json']:
        z.write(p,arcname='codigo_y_referencias/'+p)
print('ZIP listo:',out_zip,round(out_zip.stat().st_size/1024/1024,2),'MB')
if DESCARGAR_ZIP_AL_FINAL:
    from google.colab import files
    files.download(str(out_zip))
